In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import torch

assert torch.cuda.is_available(), "Colab 런타임을 GPU로 변경하세요."

device = torch.device("cuda")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

GPU: NVIDIA A100-SXM4-40GB
CUDA: 12.8


In [ ]:
import json
import time
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision.models import (
    resnet18,
    ResNet18_Weights,
)
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/YardOps_PPE_Dataset"
)

FULL_PERSON_ROI_ROOT = (
    PROJECT_ROOT
    / "shared_backbone_workspace"
    / "manifests"
    / "person_roi"
    / "full"
)

FULL_TRAIN_MANIFEST = (
    FULL_PERSON_ROI_ROOT
    / "train_hybrid_person_roi.jsonl"
)

FULL_VAL_MANIFEST = (
    FULL_PERSON_ROI_ROOT
    / "val_hybrid_person_roi.jsonl"
)

FULL_TEST_MANIFEST = (
    FULL_PERSON_ROI_ROOT
    / "test_hybrid_person_roi.jsonl"
)

for path in [
    FULL_TRAIN_MANIFEST,
    FULL_VAL_MANIFEST,
    FULL_TEST_MANIFEST,
]:
    print(path, path.exists())

/content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/person_roi/full/train_hybrid_person_roi.jsonl True
/content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/person_roi/full/val_hybrid_person_roi.jsonl True
/content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/person_roi/full/test_hybrid_person_roi.jsonl True


In [ ]:
def find_split_file(root, split):
    candidates = [
        path
        for path in root.rglob("*.jsonl")
        if split in path.name.lower()
        and "unmatched" not in path.name.lower()
        and "mini" not in path.name.lower()
    ]

    print(f"\n[{split}] 후보")

    for path in candidates:
        print(repr(str(path)))

    if not candidates:
        raise FileNotFoundError(
            f"{split} manifest를 찾지 못했습니다."
        )

    return candidates[0]


FULL_TRAIN_MANIFEST = find_split_file(
    PERSON_ROI_ROOT,
    "train",
)

FULL_VAL_MANIFEST = find_split_file(
    PERSON_ROI_ROOT,
    "val",
)

FULL_TEST_MANIFEST = find_split_file(
    PERSON_ROI_ROOT,
    "test",
)

print("\n선택 파일")
print("Train:", FULL_TRAIN_MANIFEST)
print("Val:", FULL_VAL_MANIFEST)
print("Test:", FULL_TEST_MANIFEST)


[train] 후보
'/content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/person_roi/full/train_hybrid_person_roi.jsonl'

[val] 후보
'/content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/person_roi/full/val_hybrid_person_roi.jsonl'

[test] 후보
'/content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/person_roi/full/test_hybrid_person_roi.jsonl'

선택 파일
Train: /content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/person_roi/full/train_hybrid_person_roi.jsonl
Val: /content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/person_roi/full/val_hybrid_person_roi.jsonl
Test: /content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/person_roi/full/test_hybrid_person_roi.jsonl


In [ ]:
all_jsonl = list(PROJECT_ROOT.rglob("*.jsonl"))

for path in all_jsonl:
    name = path.name.lower()

    if any(word in name for word in ["train", "val", "test"]):
        print(path)

/content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/splits/test_annotation_manifest.jsonl
/content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/splits/val_annotation_manifest.jsonl
/content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/splits/train_annotation_manifest.jsonl
/content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/person_roi/mini_hybrid_train.jsonl
/content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/person_roi/mini_hybrid_val.jsonl
/content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/person_roi/mini_hybrid_test.jsonl
/content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/splits/mini/mini_val_manifest.jsonl
/content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/splits/mini/mini_test_manifest.jsonl
/content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/

In [ ]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/YardOps_PPE_Dataset"
)

FULL_PERSON_ROI_ROOT = (
    PROJECT_ROOT
    / "shared_backbone_workspace"
    / "manifests"
    / "person_roi"
    / "full"
)

FULL_TRAIN_MANIFEST = (
    FULL_PERSON_ROI_ROOT
    / "train_hybrid_person_roi.jsonl"
)

FULL_VAL_MANIFEST = (
    FULL_PERSON_ROI_ROOT
    / "val_hybrid_person_roi.jsonl"
)

FULL_TEST_MANIFEST = (
    FULL_PERSON_ROI_ROOT
    / "test_hybrid_person_roi.jsonl"
)

FULL_CHECKPOINT_ROOT = (
    PROJECT_ROOT
    / "shared_backbone_workspace"
    / "checkpoints"
    / "full_hybrid_person_roi"
)

FULL_CHECKPOINT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

for path in [
    FULL_TRAIN_MANIFEST,
    FULL_VAL_MANIFEST,
    FULL_TEST_MANIFEST,
]:
    assert path.exists(), f"Manifest 없음: {path}"

print("Train manifest:", FULL_TRAIN_MANIFEST)
print("Val manifest:", FULL_VAL_MANIFEST)
print("Test manifest:", FULL_TEST_MANIFEST)
print("Checkpoint:", FULL_CHECKPOINT_ROOT)

Train manifest: /content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/person_roi/full/train_hybrid_person_roi.jsonl
Val manifest: /content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/person_roi/full/val_hybrid_person_roi.jsonl
Test manifest: /content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/person_roi/full/test_hybrid_person_roi.jsonl
Checkpoint: /content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/checkpoints/full_hybrid_person_roi


In [ ]:
DRIVE_DATA_ROOT = (
    PROJECT_ROOT
    / "081_스마트야드_압축해제"
    / "3.개방데이터"
    / "1.데이터"
)

assert DRIVE_DATA_ROOT.exists(), (
    f"Drive 원본 데이터 경로 없음: {DRIVE_DATA_ROOT}"
)

print("Drive data root:", DRIVE_DATA_ROOT)

Drive data root: /content/drive/MyDrive/YardOps_PPE_Dataset/081_스마트야드_압축해제/3.개방데이터/1.데이터


In [ ]:
def resolve_image_path(original_path):
    original_path = str(original_path)

    # 기존 경로가 아직 유효하면 그대로 사용
    if Path(original_path).exists():
        return original_path

    normalized = original_path.replace("\\", "/")
    marker = "/1.데이터/"

    # 1.데이터 이하 상대경로를 Drive 경로에 붙임
    if marker in normalized:
        relative_path = normalized.split(
            marker,
            maxsplit=1,
        )[1]

        candidate = (
            DRIVE_DATA_ROOT
            / relative_path
        )

        if candidate.exists():
            return str(candidate)

    return original_path


def read_jsonl_with_resolved_paths(path):
    records = []

    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as file:
        for line in file:
            line = line.strip()

            if not line:
                continue

            record = json.loads(line)

            record["image_path"] = (
                resolve_image_path(
                    record["image_path"]
                )
            )

            records.append(record)

    return records

In [ ]:
train_records = read_jsonl_with_resolved_paths(
    FULL_TRAIN_MANIFEST
)

val_records = read_jsonl_with_resolved_paths(
    FULL_VAL_MANIFEST
)

test_records = read_jsonl_with_resolved_paths(
    FULL_TEST_MANIFEST
)

print("Train:", len(train_records))
print("Val:", len(val_records))
print("Test:", len(test_records))

print("\n첫 이미지 경로:")
print(train_records[0]["image_path"])
print("존재 여부:", Path(train_records[0]["image_path"]).exists())

Train: 53672
Val: 10052
Test: 8501

첫 이미지 경로:
/content/drive/MyDrive/YardOps_PPE_Dataset/081_스마트야드_압축해제/3.개방데이터/1.데이터/Training/01.원천데이터/TS_안전장비-S63_DATA3_안전대미착용-B0/S63_DATA3_B0_L1_D2023-08-25-10-50_001_000254.jpg
존재 여부: True


In [5]:
IMAGE_SIZE = 512

TASK_TO_INDEX = {
    "helmet": 0,
    "harness": 1,
    "welding_mask": 2,
}

INDEX_TO_TASK = {
    index: task
    for task, index in TASK_TO_INDEX.items()
}


class PPEPersonROIDataset(Dataset):
    def __init__(
        self,
        records,
        image_size=512,
        augment=False,
    ):
        self.records = records
        self.image_size = image_size
        self.augment = augment

        if not self.records:
            raise ValueError("Dataset records가 비어 있습니다.")

    def __len__(self):
        return len(self.records)

    def _make_full_mask(self, record):
        height = int(record["image_height"])
        width = int(record["image_width"])

        mask = np.zeros(
            (height, width),
            dtype=np.uint8,
        )

        polygon = np.asarray(
            record["polygon"],
            dtype=np.float32,
        )

        # polygon이 flat list인 경우 처리
        if polygon.ndim == 1:
            polygon = polygon.reshape(-1, 2)

        polygon = np.round(
            polygon
        ).astype(np.int32)

        cv2.fillPoly(
            mask,
            [polygon],
            color=1,
        )

        return mask

    def _expand_person_box(
        self,
        bbox,
        image_width,
        image_height,
    ):
        x1, y1, x2, y2 = map(float, bbox)

        bbox_width = max(x2 - x1, 1.0)
        bbox_height = max(y2 - y1, 1.0)

        if self.augment:
            horizontal_ratio = np.random.uniform(
                0.12,
                0.25,
            )
            top_ratio = np.random.uniform(
                0.15,
                0.30,
            )
            bottom_ratio = np.random.uniform(
                0.08,
                0.18,
            )

            shift_x = np.random.uniform(
                -0.05,
                0.05,
            ) * bbox_width

            shift_y = np.random.uniform(
                -0.04,
                0.04,
            ) * bbox_height
        else:
            horizontal_ratio = 0.18
            top_ratio = 0.22
            bottom_ratio = 0.12
            shift_x = 0.0
            shift_y = 0.0

        x1 = (
            x1
            - bbox_width * horizontal_ratio
            + shift_x
        )

        x2 = (
            x2
            + bbox_width * horizontal_ratio
            + shift_x
        )

        y1 = (
            y1
            - bbox_height * top_ratio
            + shift_y
        )

        y2 = (
            y2
            + bbox_height * bottom_ratio
            + shift_y
        )

        x1 = max(int(round(x1)), 0)
        y1 = max(int(round(y1)), 0)
        x2 = min(int(round(x2)), image_width)
        y2 = min(int(round(y2)), image_height)

        return x1, y1, x2, y2

    def _augment_image(
        self,
        image,
        mask,
    ):
        if np.random.rand() < 0.5:
            image = np.ascontiguousarray(
                image[:, ::-1]
            )
            mask = np.ascontiguousarray(
                mask[:, ::-1]
            )

        if np.random.rand() < 0.5:
            brightness = np.random.uniform(
                0.80,
                1.20,
            )

            image = np.clip(
                image.astype(np.float32)
                * brightness,
                0,
                255,
            ).astype(np.uint8)

        return image, mask

    def __getitem__(self, index):
        record = self.records[index]

        image_path = record["image_path"]

        image_bgr = cv2.imread(image_path)

        if image_bgr is None:
            raise FileNotFoundError(
                f"이미지 로드 실패: {image_path}"
            )

        image_rgb = cv2.cvtColor(
            image_bgr,
            cv2.COLOR_BGR2RGB,
        )

        full_mask = self._make_full_mask(
            record
        )

        image_height, image_width = (
            image_rgb.shape[:2]
        )

        x1, y1, x2, y2 = (
            self._expand_person_box(
                bbox=record["person_bbox"],
                image_width=image_width,
                image_height=image_height,
            )
        )

        if x2 <= x1 or y2 <= y1:
            raise ValueError(
                f"잘못된 person bbox: "
                f"{record.get('record_id')}"
            )

        roi_image = image_rgb[
            y1:y2,
            x1:x2,
        ]

        roi_mask = full_mask[
            y1:y2,
            x1:x2,
        ]

        roi_image = cv2.resize(
            roi_image,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_LINEAR,
        )

        roi_mask = cv2.resize(
            roi_mask,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_NEAREST,
        )

        if self.augment:
            roi_image, roi_mask = (
                self._augment_image(
                    roi_image,
                    roi_mask,
                )
            )

        image_tensor = (
            torch.from_numpy(
                roi_image.copy()
            )
            .permute(2, 0, 1)
            .float()
            / 255.0
        )

        image_tensor = TF.normalize(
            image_tensor,
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        )

        mask_tensor = (
            torch.from_numpy(
                roi_mask.copy()
            )
            .float()
            .unsqueeze(0)
        )

        mask_tensor = (
            mask_tensor > 0.5
        ).float()

        return {
            "image": image_tensor,
            "mask": mask_tensor,

            "task_index": torch.tensor(
                TASK_TO_INDEX[
                    record["task_id"]
                ],
                dtype=torch.long,
            ),

            "status_label": torch.tensor(
                int(record["status_label"]),
                dtype=torch.long,
            ),

            "class_name":
                record["class_name"],

            "record_id":
                record.get(
                    "record_id",
                    str(index),
                ),
        }

In [38]:
full_train_dataset = PPEPersonROIDataset(
    records=train_records,
    image_size=IMAGE_SIZE,
    augment=True,
)

full_val_dataset = PPEPersonROIDataset(
    records=val_records,
    image_size=IMAGE_SIZE,
    augment=False,
)

full_test_dataset = PPEPersonROIDataset(
    records=test_records,
    image_size=IMAGE_SIZE,
    augment=False,
)

print("Train:", len(full_train_dataset))
print("Val:", len(full_val_dataset))
print("Test:", len(full_test_dataset))

Train: 53672
Val: 10052
Test: 8501


In [39]:
FULL_BATCH_SIZE = 32
NUM_WORKERS = 4

full_train_loader = DataLoader(
    full_train_dataset,
    batch_size=FULL_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    drop_last=True,
)

full_val_loader = DataLoader(
    full_val_dataset,
    batch_size=FULL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

full_test_loader = DataLoader(
    full_test_dataset,
    batch_size=FULL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

print("DataLoader 생성 완료")

DataLoader 생성 완료


In [ ]:
start_time = time.time()

sample_batch = next(
    iter(full_train_loader)
)

print(
    "첫 배치 로딩 시간:",
    f"{time.time() - start_time:.2f}초",
)

print(
    "image:",
    sample_batch["image"].shape,
)

print(
    "mask:",
    sample_batch["mask"].shape,
)

첫 배치 로딩 시간: 8.96초
image: torch.Size([32, 3, 512, 512])
mask: torch.Size([32, 1, 512, 512])


In [6]:
class SharedBackbonePPEModel(nn.Module):
    def __init__(
        self,
        fpn_channels=128,
        pretrained=True,
        num_tasks=3,
    ):
        super().__init__()

        weights = (
            ResNet18_Weights.DEFAULT
            if pretrained
            else None
        )

        backbone = resnet18(
            weights=weights
        )

        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
        )

        self.layer1 = backbone.layer1   # 64
        self.layer2 = backbone.layer2   # 128
        self.layer3 = backbone.layer3   # 256
        self.layer4 = backbone.layer4   # 512

        self.lateral2 = nn.Conv2d(
            64,
            fpn_channels,
            kernel_size=1,
        )

        self.lateral3 = nn.Conv2d(
            128,
            fpn_channels,
            kernel_size=1,
        )

        self.lateral4 = nn.Conv2d(
            256,
            fpn_channels,
            kernel_size=1,
        )

        self.lateral5 = nn.Conv2d(
            512,
            fpn_channels,
            kernel_size=1,
        )

        self.smooth2 = nn.Conv2d(
            fpn_channels,
            fpn_channels,
            kernel_size=3,
            padding=1,
        )

        self.segmentation_heads = (
            nn.ModuleList([
                nn.Sequential(
                    nn.Conv2d(
                        fpn_channels,
                        fpn_channels,
                        kernel_size=3,
                        padding=1,
                    ),
                    nn.BatchNorm2d(
                        fpn_channels
                    ),
                    nn.ReLU(inplace=True),

                    nn.Conv2d(
                        fpn_channels,
                        64,
                        kernel_size=3,
                        padding=1,
                    ),
                    nn.ReLU(inplace=True),

                    nn.Conv2d(
                        64,
                        1,
                        kernel_size=1,
                    ),
                )
                for _ in range(num_tasks)
            ])
        )

        self.global_pool = (
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.status_heads = (
            nn.ModuleList([
                nn.Sequential(
                    nn.Linear(512, 128),
                    nn.ReLU(inplace=True),
                    nn.Dropout(0.2),
                    nn.Linear(128, 2),
                )
                for _ in range(num_tasks)
            ])
        )

    def forward(
    self,
    images,
    task_indices,
):
      x = self.stem(images)

      c2 = self.layer1(x)
      c3 = self.layer2(c2)
      c4 = self.layer3(c3)
      c5 = self.layer4(c4)

      p5 = self.lateral5(c5)

      p4 = (
        self.lateral4(c4)
        + F.interpolate(
            p5,
            size=c4.shape[-2:],
            mode="nearest",
        )
    )

      p3 = (
        self.lateral3(c3)
        + F.interpolate(
            p4,
            size=c3.shape[-2:],
            mode="nearest",
        )
    )

      p2 = (
        self.lateral2(c2)
        + F.interpolate(
            p3,
            size=c2.shape[-2:],
            mode="nearest",
        )
    )

      p2 = self.smooth2(p2)

      batch_size = images.shape[0]

    # 최종 출력은 float32로 통일
      seg_logits = torch.empty(
        (
            batch_size,
            1,
            images.shape[-2],
            images.shape[-1],
        ),
        device=images.device,
        dtype=torch.float32,
    )

      status_logits = torch.empty(
        (batch_size, 2),
        device=images.device,
        dtype=torch.float32,
    )

      pooled_features = (
        self.global_pool(c5)
        .flatten(1)
    )

      for task_index in range(
        len(self.segmentation_heads)
    ):
        sample_mask = (
            task_indices == task_index
        )

        if not sample_mask.any():
            continue

        task_seg_logits = (
            self.segmentation_heads[
                task_index
            ](
                p2[sample_mask]
            )
        )

        task_seg_logits = F.interpolate(
            task_seg_logits,
            size=images.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        task_status_logits = (
            self.status_heads[
                task_index
            ](
                pooled_features[
                    sample_mask
                ]
            )
        )

        # AMP 연산 결과 dtype과 관계없이 float32로 통일
        seg_logits[sample_mask] = (
            task_seg_logits.float()
        )

        status_logits[sample_mask] = (
            task_status_logits.float()
        )

      return {
        "seg_logits": seg_logits,
        "status_logits": status_logits,
    }

In [ ]:
def dice_loss(
    logits,
    targets,
    smooth=1.0,
):
    probabilities = torch.sigmoid(
        logits
    )

    probabilities = probabilities.flatten(1)
    targets = targets.flatten(1)

    intersection = (
        probabilities * targets
    ).sum(dim=1)

    denominator = (
        probabilities.sum(dim=1)
        + targets.sum(dim=1)
    )

    dice = (
        2.0 * intersection + smooth
    ) / (
        denominator + smooth
    )

    return 1.0 - dice.mean()

In [ ]:
def calculate_batch_metrics(
    seg_logits,
    masks,
    status_logits,
    status_labels,
    threshold=0.5,
):
    predictions = (
        torch.sigmoid(seg_logits)
        >= threshold
    ).float()

    predictions_flat = predictions.flatten(1)
    masks_flat = masks.flatten(1)

    intersection = (
        predictions_flat
        * masks_flat
    ).sum(dim=1)

    prediction_sum = (
        predictions_flat.sum(dim=1)
    )

    target_sum = masks_flat.sum(dim=1)

    dice = (
        2.0 * intersection + 1.0
    ) / (
        prediction_sum
        + target_sum
        + 1.0
    )

    union = (
        prediction_sum
        + target_sum
        - intersection
    )

    iou = (
        intersection + 1.0
    ) / (
        union + 1.0
    )

    status_predictions = (
        status_logits.argmax(dim=1)
    )

    accuracy = (
        status_predictions
        == status_labels
    ).float()

    return {
        "dice_sum": dice.sum().item(),
        "iou_sum": iou.sum().item(),
        "correct": accuracy.sum().item(),
        "count": masks.shape[0],
    }

In [ ]:
BCE_LOSS = nn.BCEWithLogitsLoss()
CE_LOSS = nn.CrossEntropyLoss()


def compute_loss(
    outputs,
    masks,
    status_labels,
):
    seg_logits = outputs["seg_logits"]
    status_logits = outputs[
        "status_logits"
    ]

    bce = BCE_LOSS(
        seg_logits,
        masks,
    )

    dice = dice_loss(
        seg_logits,
        masks,
    )

    segmentation_loss = (
        0.5 * bce
        + 0.5 * dice
    )

    classification_loss = CE_LOSS(
        status_logits,
        status_labels,
    )

    total_loss = (
        segmentation_loss
        + classification_loss
    )

    return {
        "loss": total_loss,
        "seg_loss": segmentation_loss,
        "cls_loss": classification_loss,
    }

In [ ]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    device,
    scaler,
):
    model.train()

    totals = {
        "loss": 0.0,
        "seg_loss": 0.0,
        "cls_loss": 0.0,
        "dice_sum": 0.0,
        "iou_sum": 0.0,
        "correct": 0.0,
        "count": 0,
    }

    progress = tqdm(
        loader,
        desc="Train",
        leave=False,
    )

    for batch in progress:
        images = batch["image"].to(
            device,
            non_blocking=True,
        )

        masks = batch["mask"].to(
            device,
            non_blocking=True,
        )

        task_indices = batch[
            "task_index"
        ].to(
            device,
            non_blocking=True,
        )

        status_labels = batch[
            "status_label"
        ].to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=True,
        ):
            outputs = model(
                images,
                task_indices,
            )

            losses = compute_loss(
                outputs,
                masks,
                status_labels,
            )

        scaler.scale(
            losses["loss"]
        ).backward()

        scaler.step(optimizer)
        scaler.update()

        metrics = calculate_batch_metrics(
            outputs["seg_logits"].detach(),
            masks,
            outputs[
                "status_logits"
            ].detach(),
            status_labels,
        )

        batch_size = images.shape[0]

        totals["loss"] += (
            losses["loss"].item()
            * batch_size
        )

        totals["seg_loss"] += (
            losses["seg_loss"].item()
            * batch_size
        )

        totals["cls_loss"] += (
            losses["cls_loss"].item()
            * batch_size
        )

        totals["dice_sum"] += (
            metrics["dice_sum"]
        )

        totals["iou_sum"] += (
            metrics["iou_sum"]
        )

        totals["correct"] += (
            metrics["correct"]
        )

        totals["count"] += batch_size

        progress.set_postfix(
            loss=f"{losses['loss'].item():.4f}"
        )

    count = totals["count"]

    return {
        "loss": totals["loss"] / count,
        "seg_loss":
            totals["seg_loss"] / count,
        "cls_loss":
            totals["cls_loss"] / count,
        "dice":
            totals["dice_sum"] / count,
        "iou":
            totals["iou_sum"] / count,
        "accuracy":
            totals["correct"] / count,
    }

In [ ]:
@torch.no_grad()
def validate_one_epoch(
    model,
    loader,
    device,
):
    model.eval()

    totals = {
        "loss": 0.0,
        "seg_loss": 0.0,
        "cls_loss": 0.0,
        "dice_sum": 0.0,
        "iou_sum": 0.0,
        "correct": 0.0,
        "count": 0,
    }

    progress = tqdm(
        loader,
        desc="Validation",
        leave=False,
    )

    for batch in progress:
        images = batch["image"].to(
            device,
            non_blocking=True,
        )

        masks = batch["mask"].to(
            device,
            non_blocking=True,
        )

        task_indices = batch[
            "task_index"
        ].to(
            device,
            non_blocking=True,
        )

        status_labels = batch[
            "status_label"
        ].to(
            device,
            non_blocking=True,
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=True,
        ):
            outputs = model(
                images,
                task_indices,
            )

            losses = compute_loss(
                outputs,
                masks,
                status_labels,
            )

        metrics = calculate_batch_metrics(
            outputs["seg_logits"],
            masks,
            outputs["status_logits"],
            status_labels,
        )

        batch_size = images.shape[0]

        totals["loss"] += (
            losses["loss"].item()
            * batch_size
        )

        totals["seg_loss"] += (
            losses["seg_loss"].item()
            * batch_size
        )

        totals["cls_loss"] += (
            losses["cls_loss"].item()
            * batch_size
        )

        totals["dice_sum"] += (
            metrics["dice_sum"]
        )

        totals["iou_sum"] += (
            metrics["iou_sum"]
        )

        totals["correct"] += (
            metrics["correct"]
        )

        totals["count"] += batch_size

    count = totals["count"]

    return {
        "loss": totals["loss"] / count,
        "seg_loss":
            totals["seg_loss"] / count,
        "cls_loss":
            totals["cls_loss"] / count,
        "dice":
            totals["dice_sum"] / count,
        "iou":
            totals["iou_sum"] / count,
        "accuracy":
            totals["correct"] / count,
    }

In [ ]:
del full_model

torch.cuda.empty_cache()

full_model = SharedBackbonePPEModel(
    fpn_channels=128,
    pretrained=True,
).to(device)

In [ ]:
full_model = SharedBackbonePPEModel(
    fpn_channels=128,
    pretrained=True,
).to(device)

FULL_EPOCHS = 20
FULL_LEARNING_RATE = 1e-4
FULL_WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 4

full_optimizer = torch.optim.AdamW(
    full_model.parameters(),
    lr=FULL_LEARNING_RATE,
    weight_decay=FULL_WEIGHT_DECAY,
)

full_scheduler = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(
        full_optimizer,
        mode="min",
        factor=0.5,
        patience=1,
        min_lr=1e-6,
    )
)

full_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True,
)

print(
    "Model device:",
    next(full_model.parameters()).device,
)

Model device: cuda:0


In [ ]:
del full_model

torch.cuda.empty_cache()

full_model = SharedBackbonePPEModel(
    fpn_channels=128,
    pretrained=True,
).to(device)
full_optimizer = torch.optim.AdamW(
    full_model.parameters(),
    lr=FULL_LEARNING_RATE,
    weight_decay=FULL_WEIGHT_DECAY,
)

full_scheduler = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(
        full_optimizer,
        mode="min",
        factor=0.5,
        patience=1,
        min_lr=1e-6,
    )
)

full_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True,
)

In [ ]:
LAST_CHECKPOINT_PATH = (
    FULL_CHECKPOINT_ROOT
    / "last_checkpoint.pt"
)

START_EPOCH = 1
best_val_loss = float("inf")
best_val_dice = -1.0
loss_no_improve_count = 0
full_history = []


if LAST_CHECKPOINT_PATH.exists():
    print(
        "기존 checkpoint 발견:",
        LAST_CHECKPOINT_PATH,
    )

    resume_checkpoint = torch.load(
        LAST_CHECKPOINT_PATH,
        map_location=device,
    )

    try:
        full_model.load_state_dict(
            resume_checkpoint[
                "model_state_dict"
            ]
        )

        full_optimizer.load_state_dict(
            resume_checkpoint[
                "optimizer_state_dict"
            ]
        )

        full_scheduler.load_state_dict(
            resume_checkpoint[
                "scheduler_state_dict"
            ]
        )

        full_scaler.load_state_dict(
            resume_checkpoint[
                "scaler_state_dict"
            ]
        )

        full_history = (
            resume_checkpoint.get(
                "history",
                [],
            )
        )

        START_EPOCH = (
            resume_checkpoint["epoch"] + 1
        )

        if full_history:
            best_val_loss = min(
                row["val_loss"]
                for row in full_history
            )

            best_val_dice = max(
                row["val_dice"]
                for row in full_history
            )

            last_best_epoch = min(
                full_history,
                key=lambda row:
                    row["val_loss"],
            )["epoch"]

            loss_no_improve_count = (
                resume_checkpoint["epoch"]
                - last_best_epoch
            )

        print(
            "학습 재개 epoch:",
            START_EPOCH,
        )

        print(
            "기존 best loss:",
            best_val_loss,
        )

        print(
            "기존 best Dice:",
            best_val_dice,
        )

    except RuntimeError as error:
        print(
            "기존 checkpoint와 현재 모델 구조가 "
            "호환되지 않아 새로 학습합니다."
        )

        print("오류:", error)

        START_EPOCH = 1
        best_val_loss = float("inf")
        best_val_dice = -1.0
        loss_no_improve_count = 0
        full_history = []

else:
    print("기존 checkpoint 없음: 새 학습 시작")

기존 checkpoint 발견: /content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/checkpoints/full_hybrid_person_roi/last_checkpoint.pt
기존 checkpoint와 현재 모델 구조가 호환되지 않아 새로 학습합니다.
오류: Error(s) in loading state_dict for SharedBackbonePPEModel:
	Missing key(s) in state_dict: "lateral5.weight", "lateral5.bias", "smooth2.weight", "smooth2.bias", "segmentation_heads.0.0.weight", "segmentation_heads.0.0.bias", "segmentation_heads.0.1.running_mean", "segmentation_heads.0.1.running_var", "segmentation_heads.0.3.weight", "segmentation_heads.0.3.bias", "segmentation_heads.0.5.weight", "segmentation_heads.0.5.bias", "segmentation_heads.1.0.weight", "segmentation_heads.1.0.bias", "segmentation_heads.1.1.running_mean", "segmentation_heads.1.1.running_var", "segmentation_heads.1.3.weight", "segmentation_heads.1.3.bias", "segmentation_heads.1.5.weight", "segmentation_heads.1.5.bias", "segmentation_heads.2.0.weight", "segmentation_heads.2.0.bias", "segmentation_heads.2.1.running_mean", "segmentati

In [ ]:
test_batch = next(
    iter(full_train_loader)
)

test_images = test_batch["image"].to(
    device,
    non_blocking=True,
)

test_task_indices = test_batch[
    "task_index"
].to(
    device,
    non_blocking=True,
)

full_model.train()

with torch.amp.autocast(
    device_type="cuda",
    enabled=True,
):
    test_outputs = full_model(
        test_images,
        test_task_indices,
    )

torch.cuda.synchronize()

print(
    "Forward 성공:",
    test_outputs["seg_logits"].shape,
)

print(
    "GPU allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f}GB",
)

del test_batch
del test_images
del test_task_indices
del test_outputs

torch.cuda.empty_cache()

Forward 성공: torch.Size([32, 1, 512, 512])
GPU allocated: 7.86GB


In [ ]:
training_start_time = time.time()


for epoch in range(
    START_EPOCH,
    FULL_EPOCHS + 1,
):
    epoch_start_time = time.time()

    print("\n" + "=" * 90)
    print(
        f"Full Hybrid Person ROI "
        f"Epoch {epoch}/{FULL_EPOCHS}"
    )
    print("=" * 90)

    train_metrics = train_one_epoch(
        model=full_model,
        loader=full_train_loader,
        optimizer=full_optimizer,
        device=device,
        scaler=full_scaler,
    )

    val_metrics = validate_one_epoch(
        model=full_model,
        loader=full_val_loader,
        device=device,
    )

    full_scheduler.step(
        val_metrics["loss"]
    )

    current_lr = (
        full_optimizer
        .param_groups[0]["lr"]
    )

    epoch_seconds = (
        time.time() - epoch_start_time
    )

    history_row = {
        "epoch": epoch,
        "learning_rate": current_lr,

        "train_loss":
            train_metrics["loss"],
        "train_seg_loss":
            train_metrics["seg_loss"],
        "train_cls_loss":
            train_metrics["cls_loss"],
        "train_dice":
            train_metrics["dice"],
        "train_iou":
            train_metrics["iou"],
        "train_accuracy":
            train_metrics["accuracy"],

        "val_loss":
            val_metrics["loss"],
        "val_seg_loss":
            val_metrics["seg_loss"],
        "val_cls_loss":
            val_metrics["cls_loss"],
        "val_dice":
            val_metrics["dice"],
        "val_iou":
            val_metrics["iou"],
        "val_accuracy":
            val_metrics["accuracy"],

        "epoch_seconds": epoch_seconds,
    }

    full_history.append(history_row)

    print(
        f"Train | "
        f"loss={train_metrics['loss']:.4f}, "
        f"seg={train_metrics['seg_loss']:.4f}, "
        f"cls={train_metrics['cls_loss']:.4f}, "
        f"dice={train_metrics['dice']:.4f}, "
        f"iou={train_metrics['iou']:.4f}, "
        f"acc={train_metrics['accuracy']:.4f}"
    )

    print(
        f"Val   | "
        f"loss={val_metrics['loss']:.4f}, "
        f"seg={val_metrics['seg_loss']:.4f}, "
        f"cls={val_metrics['cls_loss']:.4f}, "
        f"dice={val_metrics['dice']:.4f}, "
        f"iou={val_metrics['iou']:.4f}, "
        f"acc={val_metrics['accuracy']:.4f}"
    )

    print(
        f"LR={current_lr:.2e} | "
        f"시간={epoch_seconds / 60:.2f}분"
    )

    checkpoint = {
        "epoch": epoch,

        "model_state_dict":
            full_model.state_dict(),

        "optimizer_state_dict":
            full_optimizer.state_dict(),

        "scheduler_state_dict":
            full_scheduler.state_dict(),

        "scaler_state_dict":
            full_scaler.state_dict(),

        "history": full_history,

        "task_to_index":
            TASK_TO_INDEX,

        "image_size":
            IMAGE_SIZE,

        "batch_size":
            FULL_BATCH_SIZE,

        "input_mode":
            "full_hybrid_person_roi",

        "person_detection_strategy": {
            "default":
                "yolo26n_person",

            "welding_mask_on_fallback":
                "sh_segmentation_class_14",
        },
    }

    # 런타임 종료 시 재개용
    torch.save(
        checkpoint,
        LAST_CHECKPOINT_PATH,
    )

    loss_improved = (
        val_metrics["loss"]
        < best_val_loss
    )

    if loss_improved:
        best_val_loss = (
            val_metrics["loss"]
        )

        loss_no_improve_count = 0

        checkpoint[
            "best_val_loss"
        ] = best_val_loss

        torch.save(
            checkpoint,
            FULL_CHECKPOINT_ROOT
            / "best_loss_model.pt",
        )

        print(
            "Best loss 저장:",
            f"{best_val_loss:.4f}",
        )

    else:
        loss_no_improve_count += 1

    if (
        val_metrics["dice"]
        > best_val_dice
    ):
        best_val_dice = (
            val_metrics["dice"]
        )

        checkpoint[
            "best_val_dice"
        ] = best_val_dice

        torch.save(
            checkpoint,
            FULL_CHECKPOINT_ROOT
            / "best_dice_model.pt",
        )

        print(
            "Best Dice 저장:",
            f"{best_val_dice:.4f}",
        )

    history_df = pd.DataFrame(
        full_history
    )

    history_df.to_csv(
        FULL_CHECKPOINT_ROOT
        / "training_history.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print(
        "Early stopping:",
        f"{loss_no_improve_count}/"
        f"{EARLY_STOPPING_PATIENCE}",
    )

    if (
        loss_no_improve_count
        >= EARLY_STOPPING_PATIENCE
    ):
        print("Early stopping 실행")
        break


total_training_time = (
    time.time() - training_start_time
)

print("\n" + "=" * 90)
print("전체 학습 종료")
print("=" * 90)

print(
    "학습 시간:",
    f"{total_training_time / 60:.2f}분",
)

print(
    "Best validation loss:",
    f"{best_val_loss:.4f}",
)

print(
    "Best validation Dice:",
    f"{best_val_dice:.4f}",
)

display(
    pd.DataFrame(full_history)
)


Full Hybrid Person ROI Epoch 1/20


Train:   0%|          | 0/1677 [00:00<?, ?it/s]

Validation:   0%|          | 0/315 [00:00<?, ?it/s]

Train | loss=0.1025, seg=0.0886, cls=0.0139, dice=0.8815, iou=0.8097, acc=0.9955
Val   | loss=0.3579, seg=0.1010, cls=0.2569, dice=0.8575, iou=0.7740, acc=0.9182
LR=1.00e-04 | 시간=58.41분
Best loss 저장: 0.3579
Best Dice 저장: 0.8575
Early stopping: 0/4

Full Hybrid Person ROI Epoch 2/20


Train:   0%|          | 0/1677 [00:00<?, ?it/s]

...

Train | loss=0.0356, seg=0.0356, cls=0.0000, dice=0.9499, iou=0.9074, acc=1.0000
Val   | loss=0.5858, seg=0.0881, cls=0.4977, dice=0.8823, iou=0.8057, acc=0.8761
LR=2.50e-05 | 시간=7.60분
Early stopping: 3/4

Full Hybrid Person ROI Epoch 8/20


Train:   0%|          | 0/1677 [00:00<?, ?it/s]

Validation:   0%|          | 0/315 [00:00<?, ?it/s]

Train | loss=0.0345, seg=0.0345, cls=0.0000, dice=0.9514, iou=0.9099, acc=1.0000
Val   | loss=0.5348, seg=0.0863, cls=0.4485, dice=0.8844, iou=0.8100, acc=0.8777
LR=1.25e-05 | 시간=7.55분
Early stopping: 4/4
Early stopping 실행

전체 학습 종료
학습 시간: 112.07분
Best validation loss: 0.3555
Best validation Dice: 0.8849


,epoch,learning_rate,train_loss,train_seg_loss,train_cls_loss,train_dice,train_iou,train_accuracy,val_loss,val_seg_loss,val_cls_loss,val_dice,val_iou,val_accuracy,epoch_seconds
0,1,0.000100,0.102508,0.088620,0.013888,0.881517,0.809671,0.995490,0.357928,0.100997,0.256931,0.857492,0.774020,0.918225,3504.716814
1,2,0.000100,0.054970,0.053923,0.001047,0.925351,0.868131,0.999870,0.470905,0.092661,0.378244,0.872839,0.793549,0.833764,460.999105
2,3,0.000050,0.051048,0.049419,0.001629,0.931288,0.877522,0.999609,1.068221,0.109015,0.959207,0.851658,0.761822,0.857441,455.765573
3,4,0.000050,0.043423,0.043229,0.000194,0.939676,0.890750,0.999963,0.355457,0.084829,0.270628,0.884202,0.810954,0.918822,453.162880
4,5,0.000050,0.040061,0.040037,0.000024,0.943883,0.897497,1.000000,0.651074,0.087088,0.563986,0.881727,0.806646,0.868782,460.103599
5,6,0.000025,0.038382,0.038367,0.000016,0.946128,0.901203,1.000000,0.422447,0.085070,0.337376,0.884870,0.811715,0.894349,452.916445
6,7,0.000025,0.035614,0.035584,0.000030,0.949869,0.907424,0.999981,0.585760,0.088102,0.497658,0.882299,0.805741,0.876144,455.941149
7,8,0.000013,0.034497,0.034490,0.000008,0.951369,0.909934,1.000000,0.534818,0.086347,0.448471,0.884443,0.809958,0.877736,452.769374


In [7]:
class PPEPersonROIDataset(Dataset):
    def __init__(
        self,
        records,
        image_size=512,
        augment=False,
    ):
        self.records = records
        self.image_size = image_size
        self.augment = augment

        if not self.records:
            raise ValueError("Dataset records가 비어 있습니다.")

    def __len__(self):
        return len(self.records)

    def _expand_person_box(
        self,
        bbox,
        image_width,
        image_height,
    ):
        x1, y1, x2, y2 = map(float, bbox)

        box_width = max(x2 - x1, 1.0)
        box_height = max(y2 - y1, 1.0)

        if self.augment:
            horizontal_ratio = np.random.uniform(0.12, 0.25)
            top_ratio = np.random.uniform(0.15, 0.30)
            bottom_ratio = np.random.uniform(0.08, 0.18)

            shift_x = np.random.uniform(-0.05, 0.05) * box_width
            shift_y = np.random.uniform(-0.04, 0.04) * box_height
        else:
            horizontal_ratio = 0.18
            top_ratio = 0.22
            bottom_ratio = 0.12
            shift_x = 0.0
            shift_y = 0.0

        x1 = x1 - box_width * horizontal_ratio + shift_x
        x2 = x2 + box_width * horizontal_ratio + shift_x
        y1 = y1 - box_height * top_ratio + shift_y
        y2 = y2 + box_height * bottom_ratio + shift_y

        x1 = max(int(round(x1)), 0)
        y1 = max(int(round(y1)), 0)
        x2 = min(int(round(x2)), image_width)
        y2 = min(int(round(y2)), image_height)

        return x1, y1, x2, y2

    def _make_roi_mask(
        self,
        polygon,
        x1,
        y1,
        x2,
        y2,
    ):
        polygon = np.asarray(
            polygon,
            dtype=np.float32,
        )

        if polygon.ndim == 1:
            polygon = polygon.reshape(-1, 2)

        roi_width = max(x2 - x1, 1)
        roi_height = max(y2 - y1, 1)

        # 원본 좌표 → person ROI 좌표
        polygon[:, 0] -= x1
        polygon[:, 1] -= y1

        # ROI 좌표 → 512×512 좌표
        polygon[:, 0] *= self.image_size / roi_width
        polygon[:, 1] *= self.image_size / roi_height

        polygon[:, 0] = np.clip(
            polygon[:, 0],
            0,
            self.image_size - 1,
        )

        polygon[:, 1] = np.clip(
            polygon[:, 1],
            0,
            self.image_size - 1,
        )

        polygon = np.round(
            polygon
        ).astype(np.int32)

        mask = np.zeros(
            (
                self.image_size,
                self.image_size,
            ),
            dtype=np.uint8,
        )

        cv2.fillPoly(
            mask,
            [polygon],
            color=1,
        )

        return mask

    def _augment_image(
        self,
        image,
        mask,
    ):
        if np.random.rand() < 0.5:
            image = np.ascontiguousarray(
                image[:, ::-1]
            )
            mask = np.ascontiguousarray(
                mask[:, ::-1]
            )

        if np.random.rand() < 0.5:
            brightness = np.random.uniform(
                0.80,
                1.20,
            )

            image = np.clip(
                image.astype(np.float32) * brightness,
                0,
                255,
            ).astype(np.uint8)

        return image, mask

    def __getitem__(self, index):
        record = self.records[index]

        image_path = record["image_path"]

        image_bgr = cv2.imread(
            image_path,
            cv2.IMREAD_COLOR,
        )

        if image_bgr is None:
            raise FileNotFoundError(
                f"이미지 로드 실패: {image_path}"
            )

        image_height, image_width = image_bgr.shape[:2]

        x1, y1, x2, y2 = self._expand_person_box(
            bbox=record["person_bbox"],
            image_width=image_width,
            image_height=image_height,
        )

        if x2 <= x1 or y2 <= y1:
            raise ValueError(
                f"잘못된 person bbox: "
                f"{record.get('record_id')}"
            )

        # 이미지도 person 영역만 먼저 자른 뒤 변환
        roi_bgr = image_bgr[
            y1:y2,
            x1:x2,
        ]

        roi_rgb = cv2.cvtColor(
            roi_bgr,
            cv2.COLOR_BGR2RGB,
        )

        roi_rgb = cv2.resize(
            roi_rgb,
            (
                self.image_size,
                self.image_size,
            ),
            interpolation=cv2.INTER_LINEAR,
        )

        roi_mask = self._make_roi_mask(
            polygon=record["polygon"],
            x1=x1,
            y1=y1,
            x2=x2,
            y2=y2,
        )

        if self.augment:
            roi_rgb, roi_mask = self._augment_image(
                roi_rgb,
                roi_mask,
            )

        image_tensor = (
            torch.from_numpy(
                roi_rgb.copy()
            )
            .permute(2, 0, 1)
            .float()
            .div_(255.0)
        )

        image_tensor = TF.normalize(
            image_tensor,
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        )

        mask_tensor = (
            torch.from_numpy(
                roi_mask.copy()
            )
            .float()
            .unsqueeze(0)
        )

        return {
            "image": image_tensor,
            "mask": mask_tensor,

            "task_index": torch.tensor(
                TASK_TO_INDEX[record["task_id"]],
                dtype=torch.long,
            ),

            "status_label": torch.tensor(
                int(record["status_label"]),
                dtype=torch.long,
            ),

            "class_name": record["class_name"],

            "record_id": record.get(
                "record_id",
                str(index),
            ),
        }

In [ ]:
full_train_dataset = PPEPersonROIDataset(
    records=train_records,
    image_size=512,
    augment=True,
)

full_val_dataset = PPEPersonROIDataset(
    records=val_records,
    image_size=512,
    augment=False,
)

FULL_BATCH_SIZE = 32
NUM_WORKERS = 2

full_train_loader = DataLoader(
    full_train_dataset,
    batch_size=FULL_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    drop_last=True,
)

full_val_loader = DataLoader(
    full_val_dataset,
    batch_size=FULL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

In [ ]:
import time

loader_iterator = iter(full_train_loader)

start = time.time()

for _ in range(20):
    batch = next(loader_iterator)

elapsed = time.time() - start

print("20 batch 로딩 시간:", round(elapsed, 2), "초")
print("batch당 평균:", round(elapsed / 20, 3), "초")

20 batch 로딩 시간: 77.46 초
batch당 평균: 3.873 초


In [ ]:
def benchmark_loader_workers(
    dataset,
    worker_values=(0, 1, 2, 4),
    batch_size=32,
    num_batches=10,
):
    results = []

    for workers in worker_values:
        loader_kwargs = {
            "dataset": dataset,
            "batch_size": batch_size,
            "shuffle": False,
            "num_workers": workers,
            "pin_memory": True,
            "drop_last": True,
        }

        if workers > 0:
            loader_kwargs.update({
                "persistent_workers": True,
                "prefetch_factor": 2,
            })

        loader = DataLoader(**loader_kwargs)

        start = time.time()
        iterator = iter(loader)

        for _ in range(num_batches):
            next(iterator)

        elapsed = time.time() - start

        results.append({
            "workers": workers,
            "total_seconds": elapsed,
            "seconds_per_batch": elapsed / num_batches,
        })

        del loader
        del iterator

    return pd.DataFrame(results)


worker_benchmark = benchmark_loader_workers(
    full_train_dataset
)

display(worker_benchmark)

,workers,total_seconds,seconds_per_batch
0,0,78.077727,7.807773
1,1,7.166835,0.716684
2,2,4.175849,0.417585
3,4,2.979483,0.297948


In [ ]:
FULL_BATCH_SIZE = 32
NUM_WORKERS = 4

full_train_loader = DataLoader(
    full_train_dataset,
    batch_size=FULL_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
    drop_last=True,
)

full_val_loader = DataLoader(
    full_val_dataset,
    batch_size=FULL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
    drop_last=False,
)

full_test_loader = DataLoader(
    full_test_dataset,
    batch_size=FULL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
    drop_last=False,
)

print("Train batch:", len(full_train_loader))
print("Val batch:", len(full_val_loader))
print("Test batch:", len(full_test_loader))

Train batch: 1677
Val batch: 315
Test batch: 266


In [ ]:
import time

loader_iterator = iter(full_train_loader)

# 워밍업
for _ in range(5):
    next(loader_iterator)

start_time = time.time()

for _ in range(30):
    next(loader_iterator)

elapsed = time.time() - start_time

print("30 batch:", round(elapsed, 2), "초")
print("batch당:", round(elapsed / 30, 3), "초")

30 batch: 56.36 초
batch당: 1.879 초


In [ ]:
full_model.train()

loader_iterator = iter(full_train_loader)

# DataLoader 및 CUDA 워밍업
for _ in range(3):
    batch = next(loader_iterator)

    images = batch["image"].to(
        device,
        non_blocking=True,
    )
    masks = batch["mask"].to(
        device,
        non_blocking=True,
    )
    task_indices = batch["task_index"].to(
        device,
        non_blocking=True,
    )
    status_labels = batch["status_label"].to(
        device,
        non_blocking=True,
    )

    full_optimizer.zero_grad(set_to_none=True)

    with torch.amp.autocast(
        device_type="cuda",
        enabled=True,
    ):
        outputs = full_model(
            images,
            task_indices,
        )

        losses = compute_loss(
            outputs,
            masks,
            status_labels,
        )

    full_scaler.scale(losses["loss"]).backward()
    full_scaler.step(full_optimizer)
    full_scaler.update()

torch.cuda.synchronize()

# 실제 20 step 측정
start_time = time.time()

for _ in range(20):
    batch = next(loader_iterator)

    images = batch["image"].to(
        device,
        non_blocking=True,
    )
    masks = batch["mask"].to(
        device,
        non_blocking=True,
    )
    task_indices = batch["task_index"].to(
        device,
        non_blocking=True,
    )
    status_labels = batch["status_label"].to(
        device,
        non_blocking=True,
    )

    full_optimizer.zero_grad(set_to_none=True)

    with torch.amp.autocast(
        device_type="cuda",
        enabled=True,
    ):
        outputs = full_model(
            images,
            task_indices,
        )

        losses = compute_loss(
            outputs,
            masks,
            status_labels,
        )

    full_scaler.scale(losses["loss"]).backward()
    full_scaler.step(full_optimizer)
    full_scaler.update()

torch.cuda.synchronize()

elapsed = time.time() - start_time

print("20 training steps:", round(elapsed, 2), "초")
print("step당:", round(elapsed / 20, 3), "초")
print("마지막 loss:", round(losses["loss"].item(), 4))

20 training steps: 39.54 초
step당: 1.977 초
마지막 loss: 0.1212


In [ ]:
del full_model
torch.cuda.empty_cache()

full_model = SharedBackbonePPEModel(
    fpn_channels=128,
    pretrained=True,
).to(device)

full_optimizer = torch.optim.AdamW(
    full_model.parameters(),
    lr=FULL_LEARNING_RATE,
    weight_decay=FULL_WEIGHT_DECAY,
)

full_scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        full_optimizer,
        mode="min",
        factor=0.5,
        patience=1,
        min_lr=1e-6,
    )
)

full_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True,
)

START_EPOCH = 1
best_val_loss = float("inf")
best_val_dice = -1.0
loss_no_improve_count = 0
full_history = []

print("본 학습용 모델 재초기화 완료")

본 학습용 모델 재초기화 완료


#다시 전체 학습 셀 실행

#Test: 평가

In [ ]:
@torch.no_grad()
def evaluate_checkpoint(
    checkpoint_path,
    model,
    loader,
    device,
):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    metrics = validate_one_epoch(
        model=model,
        loader=loader,
        device=device,
    )

    return {
        "checkpoint": checkpoint_path.name,
        "epoch": checkpoint["epoch"],
        **metrics,
    }


test_results = []

for checkpoint_name in [
    "best_loss_model.pt",
    "best_dice_model.pt",
]:
    checkpoint_path = (
        FULL_CHECKPOINT_ROOT
        / checkpoint_name
    )

    result = evaluate_checkpoint(
        checkpoint_path=checkpoint_path,
        model=full_model,
        loader=full_test_loader,
        device=device,
    )

    test_results.append(result)


test_result_df = pd.DataFrame(
    test_results
)

display(
    test_result_df.round(4)
)

Validation:   0%|          | 0/266 [00:00<?, ?it/s]

Validation:   0%|          | 0/266 [00:00<?, ?it/s]

,checkpoint,epoch,loss,seg_loss,cls_loss,dice,iou,accuracy
0,best_loss_model.pt,4,0.1650,0.0940,0.0709,0.8691,0.7929,0.9701
1,best_dice_model.pt,6,0.1989,0.0957,0.1032,0.8687,0.7916,0.9472


In [ ]:
FINAL_CHECKPOINT_PATH = (
    FULL_CHECKPOINT_ROOT
    / "best_loss_model.pt"
)

final_checkpoint = torch.load(
    FINAL_CHECKPOINT_PATH,
    map_location=device,
)

full_model.load_state_dict(
    final_checkpoint["model_state_dict"]
)

full_model.eval()

print("불러온 checkpoint:", FINAL_CHECKPOINT_PATH.name)
print("학습 epoch:", final_checkpoint["epoch"])

불러온 checkpoint: best_loss_model.pt
학습 epoch: 4


In [34]:
from collections import defaultdict


@torch.no_grad()
def evaluate_by_class(
    model,
    loader,
    device,
    threshold=0.5,
):
    model.eval()

    class_totals = defaultdict(
        lambda: {
            "dice_sum": 0.0,
            "iou_sum": 0.0,
            "correct": 0,
            "count": 0,
            "tp": 0,
            "fp": 0,
            "tn": 0,
            "fn": 0,
        }
    )

    for batch in tqdm(
        loader,
        desc="Class-wise test",
    ):
        images = batch["image"].to(
            device,
            non_blocking=True,
        )

        masks = batch["mask"].to(
            device,
            non_blocking=True,
        )

        task_indices = batch[
            "task_index"
        ].to(
            device,
            non_blocking=True,
        )

        status_labels = batch[
            "status_label"
        ].to(
            device,
            non_blocking=True,
        )

        class_names = batch[
            "class_name"
        ]

        with torch.amp.autocast(
            device_type="cuda",
            enabled=True,
        ):
            outputs = model(
                images,
                task_indices,
            )

        seg_predictions = (
            torch.sigmoid(
                outputs["seg_logits"]
            )
            >= threshold
        ).float()

        status_predictions = (
            outputs["status_logits"]
            .argmax(dim=1)
        )

        pred_flat = (
            seg_predictions.flatten(1)
        )

        mask_flat = masks.flatten(1)

        intersection = (
            pred_flat * mask_flat
        ).sum(dim=1)

        pred_sum = pred_flat.sum(dim=1)
        target_sum = mask_flat.sum(dim=1)

        dice_scores = (
            2.0 * intersection + 1.0
        ) / (
            pred_sum
            + target_sum
            + 1.0
        )

        union = (
            pred_sum
            + target_sum
            - intersection
        )

        iou_scores = (
            intersection + 1.0
        ) / (
            union + 1.0
        )

        for index, class_name in enumerate(
            class_names
        ):
            true_label = int(
                status_labels[index].item()
            )

            pred_label = int(
                status_predictions[index].item()
            )

            class_totals[class_name][
                "dice_sum"
            ] += float(
                dice_scores[index].item()
            )

            class_totals[class_name][
                "iou_sum"
            ] += float(
                iou_scores[index].item()
            )

            class_totals[class_name][
                "correct"
            ] += int(
                true_label == pred_label
            )

            class_totals[class_name][
                "count"
            ] += 1

            if true_label == 1 and pred_label == 1:
                class_totals[class_name]["tp"] += 1

            elif true_label == 0 and pred_label == 1:
                class_totals[class_name]["fp"] += 1

            elif true_label == 0 and pred_label == 0:
                class_totals[class_name]["tn"] += 1

            elif true_label == 1 and pred_label == 0:
                class_totals[class_name]["fn"] += 1

    rows = []

    for class_name, values in class_totals.items():
        count = values["count"]

        rows.append({
            "class_name": class_name,
            "count": count,
            "dice":
                values["dice_sum"] / count,
            "iou":
                values["iou_sum"] / count,
            "accuracy":
                values["correct"] / count,
            "tp": values["tp"],
            "fp": values["fp"],
            "tn": values["tn"],
            "fn": values["fn"],
        })

    result_df = pd.DataFrame(rows)

    class_order = [
        "harness_off",
        "harness_on",
        "helmet_off",
        "helmet_on",
        "welding_mask_off",
        "welding_mask_on",
    ]

    result_df["class_name"] = pd.Categorical(
        result_df["class_name"],
        categories=class_order,
        ordered=True,
    )

    result_df = (
        result_df
        .sort_values("class_name")
        .reset_index(drop=True)
    )

    return result_df

In [ ]:
test_class_results = evaluate_by_class(
    model=full_model,
    loader=full_test_loader,
    device=device,
)

display(
    test_class_results.round(4)
)

Class-wise test:   0%|          | 0/266 [00:00<?, ?it/s]

,class_name,count,dice,iou,accuracy,tp,fp,tn,fn
0,harness_off,1857,0.9002,0.8260,1.0000,0,0,1857,0
1,harness_on,1815,0.7638,0.6346,0.9978,1811,0,0,4
2,helmet_off,1434,0.9486,0.9037,0.9993,0,1,1433,0
3,helmet_on,1449,0.8027,0.7365,0.8282,1200,0,0,249
4,welding_mask_off,1420,0.9358,0.8813,1.0000,0,0,1420,0
5,welding_mask_on,526,0.9090,0.8376,1.0000,526,0,0,0


In [ ]:
@torch.no_grad()
def collect_status_predictions(
    model,
    loader,
    device,
):
    model.eval()

    rows = []

    for batch in tqdm(
        loader,
        desc="Collect predictions",
    ):
        images = batch["image"].to(
            device,
            non_blocking=True,
        )

        task_indices = batch[
            "task_index"
        ].to(
            device,
            non_blocking=True,
        )

        status_labels = batch[
            "status_label"
        ].to(
            device,
            non_blocking=True,
        )

        class_names = batch[
            "class_name"
        ]

        record_ids = batch[
            "record_id"
        ]

        with torch.amp.autocast(
            device_type="cuda",
            enabled=True,
        ):
            outputs = model(
                images,
                task_indices,
            )

        probabilities = torch.softmax(
            outputs["status_logits"],
            dim=1,
        )

        predictions = probabilities.argmax(
            dim=1
        )

        for index in range(
            len(class_names)
        ):
            class_name = class_names[index]

            if class_name.startswith("helmet"):
                task_name = "helmet"

            elif class_name.startswith("harness"):
                task_name = "harness"

            elif class_name.startswith(
                "welding_mask"
            ):
                task_name = "welding_mask"

            else:
                task_name = "unknown"

            rows.append({
                "record_id":
                    record_ids[index],
                "class_name":
                    class_name,
                "task_name":
                    task_name,
                "true_label":
                    int(
                        status_labels[
                            index
                        ].item()
                    ),
                "pred_label":
                    int(
                        predictions[
                            index
                        ].item()
                    ),
                "prob_off":
                    float(
                        probabilities[
                            index,
                            0,
                        ].item()
                    ),
                "prob_on":
                    float(
                        probabilities[
                            index,
                            1,
                        ].item()
                    ),
            })

    return pd.DataFrame(rows)

In [ ]:
test_prediction_df = (
    collect_status_predictions(
        model=full_model,
        loader=full_test_loader,
        device=device,
    )
)

display(
    test_prediction_df.head()
)

Collect predictions:   0%|          | 0/266 [00:00<?, ?it/s]

,record_id,class_name,task_name,true_label,pred_label,prob_off,prob_on
0,S63_DATA3_B0_L1_D2023-09-07-13-12_001_000080__...,harness_off,harness,0,0,0.999997,0.000002
1,S63_DATA3_B0_L1_D2023-09-07-13-12_001_000083__...,harness_off,harness,0,0,0.999999,0.000001
2,S63_DATA3_B0_L1_D2023-09-07-13-12_001_000089__...,harness_off,harness,0,0,0.999998,0.000002
3,S63_DATA3_B0_L1_D2023-09-07-13-12_001_000090__...,harness_off,harness,0,0,0.999998,0.000002
4,S63_DATA3_B0_L1_D2023-09-07-13-12_001_000091__...,harness_off,harness,0,0,0.999998,0.000002


In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
)


for task_name in [
    "helmet",
    "harness",
    "welding_mask",
]:
    task_df = test_prediction_df[
        test_prediction_df["task_name"]
        == task_name
    ]

    matrix = confusion_matrix(
        task_df["true_label"],
        task_df["pred_label"],
        labels=[0, 1],
    )

    print("\n" + "=" * 60)
    print(task_name)
    print("=" * 60)

    matrix_df = pd.DataFrame(
        matrix,
        index=[
            "Actual OFF",
            "Actual ON",
        ],
        columns=[
            "Pred OFF",
            "Pred ON",
        ],
    )

    display(matrix_df)

    print(
        classification_report(
            task_df["true_label"],
            task_df["pred_label"],
            labels=[0, 1],
            target_names=[
                "OFF",
                "ON",
            ],
            digits=4,
            zero_division=0,
        )
    )


helmet


,Pred OFF,Pred ON
Actual OFF,1433,1
Actual ON,249,1200


              precision    recall  f1-score   support

         OFF     0.8520    0.9993    0.9198      1434
          ON     0.9992    0.8282    0.9057      1449

    accuracy                         0.9133      2883
   macro avg     0.9256    0.9137    0.9127      2883
weighted avg     0.9259    0.9133    0.9127      2883


harness


,Pred OFF,Pred ON
Actual OFF,1857,0
Actual ON,4,1811


              precision    recall  f1-score   support

         OFF     0.9979    1.0000    0.9989      1857
          ON     1.0000    0.9978    0.9989      1815

    accuracy                         0.9989      3672
   macro avg     0.9989    0.9989    0.9989      3672
weighted avg     0.9989    0.9989    0.9989      3672


welding_mask


,Pred OFF,Pred ON
Actual OFF,1420,0
Actual ON,0,526


              precision    recall  f1-score   support

         OFF     1.0000    1.0000    1.0000      1420
          ON     1.0000    1.0000    1.0000       526

    accuracy                         1.0000      1946
   macro avg     1.0000    1.0000    1.0000      1946
weighted avg     1.0000    1.0000    1.0000      1946



In [ ]:
misclassified_df = (
    test_prediction_df[
        test_prediction_df[
            "true_label"
        ]
        != test_prediction_df[
            "pred_label"
        ]
    ]
    .copy()
)

misclassified_df[
    "confidence"
] = misclassified_df[
    ["prob_off", "prob_on"]
].max(axis=1)

misclassified_df = (
    misclassified_df
    .sort_values(
        "confidence",
        ascending=False,
    )
)

print(
    "전체 오분류:",
    len(misclassified_df),
)

display(
    misclassified_df.head(30)
)

전체 오분류: 254


,record_id,class_name,task_name,true_label,pred_label,prob_off,prob_on,confidence
5275,S63_DATA3_H1_L1_D2023-09-14-10-23_001_000739__...,helmet_on,helmet,1,0,0.998283,0.001717,0.998283
6378,S63_DATA3_H1_L1_D2023-10-12-10-16_001_001772__...,helmet_on,helmet,1,0,0.997373,0.002627,0.997373
6380,S63_DATA3_H1_L1_D2023-10-12-10-16_001_001774__...,helmet_on,helmet,1,0,0.997116,0.002884,0.997116
6379,S63_DATA3_H1_L1_D2023-10-12-10-16_001_001773__...,helmet_on,helmet,1,0,0.996008,0.003992,0.996008
5347,S63_DATA3_H1_L1_D2023-09-14-10-23_001_000850__...,helmet_on,helmet,1,0,0.993562,0.006438,0.993562
6382,S63_DATA3_H1_L1_D2023-10-12-10-16_001_001797__...,helmet_on,helmet,1,0,0.993301,0.006699,0.993301
6291,S63_DATA3_H1_L1_D2023-10-12-10-16_001_001581__...,helmet_on,helmet,1,0,0.992837,0.007163,0.992837
6377,S63_DATA3_H1_L1_D2023-10-12-10-16_001_001771__...,helmet_on,helmet,1,0,0.992640,0.007360,0.992640
6376,S63_DATA3_H1_L1_D2023-10-12-10-16_001_001770__...,helmet_on,helmet,1,0,0.988154,0.011846,0.988154
5282,S63_DATA3_H1_L1_D2023-09-14-10-23_001_000747__...,helmet_on,helmet,1,0,0.986967,0.013033,0.986967


In [ ]:
RESULT_ROOT = (
    FULL_CHECKPOINT_ROOT
    / "evaluation"
)

RESULT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

test_class_results.to_csv(
    RESULT_ROOT
    / "test_class_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

test_prediction_df.to_csv(
    RESULT_ROOT
    / "test_status_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

misclassified_df.to_csv(
    RESULT_ROOT
    / "test_misclassified.csv",
    index=False,
    encoding="utf-8-sig",
)

print("평가 결과 저장 완료")
print(RESULT_ROOT)

평가 결과 저장 완료
/content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/checkpoints/full_hybrid_person_roi/evaluation


In [ ]:
helmet_on_errors = test_prediction_df[
    (test_prediction_df["task_name"] == "helmet")
    & (test_prediction_df["true_label"] == 1)
    & (test_prediction_df["pred_label"] == 0)
].copy()

helmet_on_errors["confidence_off"] = helmet_on_errors["prob_off"]

helmet_on_errors = helmet_on_errors.sort_values(
    "confidence_off",
    ascending=False,
).reset_index(drop=True)

print("helmet_on → off 오분류:", len(helmet_on_errors))

display(
    helmet_on_errors.head(20)
)

helmet_on → off 오분류: 249


,record_id,class_name,task_name,true_label,pred_label,prob_off,prob_on,confidence_off
0,S63_DATA3_H1_L1_D2023-09-14-10-23_001_000739__...,helmet_on,helmet,1,0,0.998283,0.001717,0.998283
1,S63_DATA3_H1_L1_D2023-10-12-10-16_001_001772__...,helmet_on,helmet,1,0,0.997373,0.002627,0.997373
2,S63_DATA3_H1_L1_D2023-10-12-10-16_001_001774__...,helmet_on,helmet,1,0,0.997116,0.002884,0.997116
3,S63_DATA3_H1_L1_D2023-10-12-10-16_001_001773__...,helmet_on,helmet,1,0,0.996008,0.003992,0.996008
4,S63_DATA3_H1_L1_D2023-09-14-10-23_001_000850__...,helmet_on,helmet,1,0,0.993562,0.006438,0.993562
5,S63_DATA3_H1_L1_D2023-10-12-10-16_001_001797__...,helmet_on,helmet,1,0,0.993301,0.006699,0.993301
6,S63_DATA3_H1_L1_D2023-10-12-10-16_001_001581__...,helmet_on,helmet,1,0,0.992837,0.007163,0.992837
7,S63_DATA3_H1_L1_D2023-10-12-10-16_001_001771__...,helmet_on,helmet,1,0,0.992640,0.007360,0.992640
8,S63_DATA3_H1_L1_D2023-10-12-10-16_001_001770__...,helmet_on,helmet,1,0,0.988154,0.011846,0.988154
9,S63_DATA3_H1_L1_D2023-09-14-10-23_001_000747__...,helmet_on,helmet,1,0,0.986967,0.013033,0.986967


In [ ]:
test_record_map = {
    str(record["record_id"]): record
    for record in test_records
}

missing_record_ids = [
    record_id
    for record_id in helmet_on_errors["record_id"]
    if str(record_id) not in test_record_map
]

print("record 연결 실패:", len(missing_record_ids))

record 연결 실패: 0


In [8]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
import torch
from torchvision.transforms import functional as TF


def prepare_roi_for_visualization(
    record,
    dataset,
):
    image_bgr = cv2.imread(
        record["image_path"],
        cv2.IMREAD_COLOR,
    )

    if image_bgr is None:
        raise FileNotFoundError(
            record["image_path"]
        )

    image_rgb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB,
    )

    image_height, image_width = image_rgb.shape[:2]

    x1, y1, x2, y2 = dataset._expand_person_box(
        bbox=record["person_bbox"],
        image_width=image_width,
        image_height=image_height,
    )

    roi_rgb = image_rgb[
        y1:y2,
        x1:x2,
    ]

    roi_rgb = cv2.resize(
        roi_rgb,
        (
            dataset.image_size,
            dataset.image_size,
        ),
        interpolation=cv2.INTER_LINEAR,
    )

    gt_mask = make_roi_mask(
    polygon=record["polygon"],
    x1=x1,
    y1=y1,
    x2=x2,
    y2=y2,
    image_size=dataset.image_size,
)

    image_tensor = (
        torch.from_numpy(
            roi_rgb.copy()
        )
        .permute(2, 0, 1)
        .float()
        / 255.0
    )

    image_tensor = TF.normalize(
        image_tensor,
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )

    return {
        "full_image": image_rgb,
        "roi_image": roi_rgb,
        "gt_mask": gt_mask,
        "bbox": (x1, y1, x2, y2),
        "image_tensor": image_tensor,
    }

In [9]:
@torch.no_grad()
def predict_single_record(
    model,
    record,
    dataset,
    device,
    mask_threshold=0.5,
):
    prepared = prepare_roi_for_visualization(
        record=record,
        dataset=dataset,
    )

    image_tensor = prepared[
        "image_tensor"
    ].unsqueeze(0).to(device)

    task_index = torch.tensor(
        [
            TASK_TO_INDEX[
                record["task_id"]
            ]
        ],
        dtype=torch.long,
        device=device,
    )

    model.eval()

    with torch.amp.autocast(
        device_type="cuda",
        enabled=True,
    ):
        outputs = model(
            image_tensor,
            task_index,
        )

    probabilities = torch.softmax(
        outputs["status_logits"],
        dim=1,
    )[0]

    pred_mask = (
        torch.sigmoid(
            outputs["seg_logits"][0, 0]
        )
        >= mask_threshold
    ).cpu().numpy().astype(np.uint8)

    prepared["pred_mask"] = pred_mask
    prepared["prob_off"] = float(
        probabilities[0].item()
    )
    prepared["prob_on"] = float(
        probabilities[1].item()
    )
    prepared["pred_label"] = int(
        probabilities.argmax().item()
    )

    return prepared

In [10]:
def show_helmet_on_errors(
    error_df,
    record_map,
    model,
    dataset,
    device,
    start_index=0,
    count=12,
):
    selected = error_df.iloc[
        start_index:start_index + count
    ]

    for _, row in selected.iterrows():
        record_id = str(
            row["record_id"]
        )

        record = record_map[
            record_id
        ]

        result = predict_single_record(
            model=model,
            record=record,
            dataset=dataset,
            device=device,
        )

        full_image = result[
            "full_image"
        ].copy()

        x1, y1, x2, y2 = result["bbox"]

        cv2.rectangle(
            full_image,
            (x1, y1),
            (x2, y2),
            (255, 0, 0),
            3,
        )

        roi_image = result["roi_image"]
        gt_mask = result["gt_mask"]
        pred_mask = result["pred_mask"]

        gt_overlay = roi_image.copy()
        gt_overlay[gt_mask == 1] = (
            0.5 * gt_overlay[gt_mask == 1]
            + 0.5 * np.array([255, 255, 255])
        ).astype(np.uint8)

        pred_overlay = roi_image.copy()
        pred_overlay[pred_mask == 1] = (
            0.5 * pred_overlay[pred_mask == 1]
            + 0.5 * np.array([255, 255, 255])
        ).astype(np.uint8)

        fig, axes = plt.subplots(
            1,
            4,
            figsize=(18, 5),
        )

        axes[0].imshow(full_image)
        axes[0].set_title("Original + Person ROI")
        axes[0].axis("off")

        axes[1].imshow(roi_image)
        axes[1].set_title("Person ROI")
        axes[1].axis("off")

        axes[2].imshow(gt_overlay)
        axes[2].set_title("GT Helmet Mask")
        axes[2].axis("off")

        axes[3].imshow(pred_overlay)
        axes[3].set_title("Predicted Mask")
        axes[3].axis("off")

        fig.suptitle(
            f"{record_id}\n"
            f"True=ON, Pred=OFF | "
            f"P(OFF)={result['prob_off']:.4f}, "
            f"P(ON)={result['prob_on']:.4f}",
            fontsize=12,
        )

        plt.tight_layout()
        plt.show()

In [16]:
def make_roi_mask(
    polygon,
    x1,
    y1,
    x2,
    y2,
    image_size=512,
):
    polygon = np.asarray(
        polygon,
        dtype=np.float32,
    )

    if polygon.ndim == 1:
        polygon = polygon.reshape(-1, 2)

    polygon = polygon.copy()

    roi_width = max(x2 - x1, 1)
    roi_height = max(y2 - y1, 1)

    # 원본 이미지 좌표 → ROI 내부 좌표
    polygon[:, 0] -= x1
    polygon[:, 1] -= y1

    # ROI 좌표 → 512×512 좌표
    polygon[:, 0] *= image_size / roi_width
    polygon[:, 1] *= image_size / roi_height

    polygon[:, 0] = np.clip(
        polygon[:, 0],
        0,
        image_size - 1,
    )

    polygon[:, 1] = np.clip(
        polygon[:, 1],
        0,
        image_size - 1,
    )

    polygon = np.round(
        polygon
    ).astype(np.int32)

    mask = np.zeros(
        (image_size, image_size),
        dtype=np.uint8,
    )

    cv2.fillPoly(
        mask,
        [polygon],
        color=1,
    )

    return mask

In [ ]:
show_helmet_on_errors(
    error_df=helmet_on_errors,
    record_map=test_record_map,
    model=full_model,
    dataset=full_test_dataset,
    device=device,
    start_index=0,
    count=12,
)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
show_helmet_on_errors(
    error_df=helmet_on_errors,
    record_map=test_record_map,
    model=full_model,
    dataset=full_test_dataset,
    device=device,
    start_index=12,
    count=12,
)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
helmet_on_errors[
    "session_id"
] = helmet_on_errors[
    "record_id"
].str.extract(
    r"^(S\d+_DATA\d+)"
)

display(
    helmet_on_errors[
        "session_id"
    ]
    .value_counts()
    .head(20)
)

,count
session_id,
S63_DATA3,249


In [ ]:
from collections import Counter

def get_session_id(record_id):
    parts = str(record_id).split("_")
    return "_".join(parts[:2])

for split_name, records in [
    ("train", train_records),
    ("val", val_records),
    ("test", test_records),
]:
    counter = Counter(
        get_session_id(record["record_id"])
        for record in records
        if record["class_name"] in [
            "helmet_on",
            "helmet_off",
        ]
    )

    print(
        split_name,
        "S63_DATA3:",
        counter.get("S63_DATA3", 0),
    )

train S63_DATA3: 22342
val S63_DATA3: 3786
test S63_DATA3: 2883


In [ ]:
import pandas as pd

rows = []

for split_name, records in [
    ("train", train_records),
    ("val", val_records),
    ("test", test_records),
]:
    for record in records:
        session_id = get_session_id(
            record["record_id"]
        )

        if session_id != "S63_DATA3":
            continue

        if record["class_name"] not in [
            "helmet_on",
            "helmet_off",
        ]:
            continue

        rows.append({
            "split": split_name,
            "session_id": session_id,
            "class_name": record["class_name"],
        })

session_distribution = (
    pd.DataFrame(rows)
    .groupby(
        ["split", "class_name"]
    )
    .size()
    .reset_index(name="count")
)

display(session_distribution)

,split,class_name,count
0,test,helmet_off,1434
1,test,helmet_on,1449
2,train,helmet_off,11975
3,train,helmet_on,10367
4,val,helmet_off,1362
5,val,helmet_on,2424


In [ ]:
test_prediction_df["session_id"] = (
    test_prediction_df["record_id"]
    .astype(str)
    .str.extract(r"^(S\d+_DATA\d+)")
)

session_helmet = test_prediction_df[
    test_prediction_df["task_name"] == "helmet"
].copy()

session_metrics = []

for session_id, group in session_helmet.groupby("session_id"):
    actual_off = group[group["true_label"] == 0]
    actual_on = group[group["true_label"] == 1]

    tn = ((group["true_label"] == 0) & (group["pred_label"] == 0)).sum()
    fp = ((group["true_label"] == 0) & (group["pred_label"] == 1)).sum()
    fn = ((group["true_label"] == 1) & (group["pred_label"] == 0)).sum()
    tp = ((group["true_label"] == 1) & (group["pred_label"] == 1)).sum()

    session_metrics.append({
        "session_id": session_id,
        "helmet_off_count": len(actual_off),
        "helmet_on_count": len(actual_on),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "on_recall": tp / (tp + fn) if (tp + fn) else None,
        "accuracy": (tp + tn) / len(group),
    })

session_metrics_df = pd.DataFrame(session_metrics).sort_values(
    ["fn", "helmet_on_count"],
    ascending=False,
)

display(session_metrics_df.head(20).round(4))

,session_id,helmet_off_count,helmet_on_count,tn,fp,fn,tp,on_recall,accuracy
0,S63_DATA3,1434,1449,1433,1,249,1200,0.8282,0.9133


In [ ]:
s63_df = session_helmet[
    session_helmet["session_id"] == "S63_DATA3"
]

print("S63_DATA3 전체 helmet:", len(s63_df))
print(
    s63_df.groupby(
        ["true_label", "pred_label"]
    ).size()
)

s63_on = s63_df[
    s63_df["true_label"] == 1
]

s63_on_recall = (
    (s63_on["pred_label"] == 1).mean()
)

print(
    "S63_DATA3 helmet_on 개수:",
    len(s63_on)
)

print(
    "S63_DATA3 helmet_on recall:",
    round(s63_on_recall, 4)
)

S63_DATA3 전체 helmet: 2883
true_label  pred_label
0           0             1433
            1                1
1           0              249
            1             1200
dtype: int64
S63_DATA3 helmet_on 개수: 1449
S63_DATA3 helmet_on recall: 0.8282


In [ ]:
other_sessions = session_helmet[
    session_helmet["session_id"] != "S63_DATA3"
]

other_on = other_sessions[
    other_sessions["true_label"] == 1
]

print(
    "S63 제외 helmet_on 개수:",
    len(other_on)
)

print(
    "S63 제외 helmet_on recall:",
    round(
        (other_on["pred_label"] == 1).mean(),
        4,
    )
)

print(
    "S63 제외 helmet 전체 accuracy:",
    round(
        (
            other_sessions["true_label"]
            == other_sessions["pred_label"]
        ).mean(),
        4,
    )
)

S63 제외 helmet_on 개수: 0
S63 제외 helmet_on recall: nan
S63 제외 helmet 전체 accuracy: nan


In [ ]:
from pathlib import Path

CHECKPOINT_ROOT = Path(
    "/content/drive/MyDrive/YardOps_PPE_Dataset/"
    "shared_backbone_workspace/checkpoints/full_hybrid_person_roi"
)

for name in [
    "best_loss_model.pt",
    "best_dice_model.pt",
    "last_checkpoint.pt",
    "training_history.csv",
]:
    path = CHECKPOINT_ROOT / name
    print(name, path.exists(), path)

best_loss_model.pt True /content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/checkpoints/full_hybrid_person_roi/best_loss_model.pt
best_dice_model.pt True /content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/checkpoints/full_hybrid_person_roi/best_dice_model.pt
last_checkpoint.pt True /content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/checkpoints/full_hybrid_person_roi/last_checkpoint.pt
training_history.csv True /content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/checkpoints/full_hybrid_person_roi/training_history.csv


In [ ]:
RESULT_ROOT = CHECKPOINT_ROOT / "evaluation"

if RESULT_ROOT.exists():
    for path in RESULT_ROOT.iterdir():
        print(path.name)

test_class_metrics.csv
test_status_predictions.csv
test_misclassified.csv


##체크포인트

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [2]:
from pathlib import Path
import json
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm

assert torch.cuda.is_available()

device = torch.device("cuda")
print(torch.cuda.get_device_name(0))

NVIDIA A100-SXM4-40GB


In [3]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/YardOps_PPE_Dataset"
)

PERSON_ROI_ROOT = (
    PROJECT_ROOT
    / "shared_backbone_workspace"
    / "manifests"
    / "person_roi"
    / "full"
)

FULL_TRAIN_MANIFEST = (
    PERSON_ROI_ROOT
    / "train_hybrid_person_roi.jsonl"
)

FULL_VAL_MANIFEST = (
    PERSON_ROI_ROOT
    / "val_hybrid_person_roi.jsonl"
)

FULL_TEST_MANIFEST = (
    PERSON_ROI_ROOT
    / "test_hybrid_person_roi.jsonl"
)

FULL_CHECKPOINT_ROOT = (
    PROJECT_ROOT
    / "shared_backbone_workspace"
    / "checkpoints"
    / "full_hybrid_person_roi"
)

FINAL_CHECKPOINT_PATH = (
    FULL_CHECKPOINT_ROOT
    / "best_loss_model.pt"
)

for path in [
    FULL_TRAIN_MANIFEST,
    FULL_VAL_MANIFEST,
    FULL_TEST_MANIFEST,
    FINAL_CHECKPOINT_PATH,
]:
    print(path.exists(), path)

True /content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/person_roi/full/train_hybrid_person_roi.jsonl
True /content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/person_roi/full/val_hybrid_person_roi.jsonl
True /content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/manifests/person_roi/full/test_hybrid_person_roi.jsonl
True /content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/checkpoints/full_hybrid_person_roi/best_loss_model.pt


In [4]:
DRIVE_DATA_ROOT = (
    PROJECT_ROOT
    / "081_스마트야드_압축해제"
    / "3.개방데이터"
    / "1.데이터"
)

def resolve_image_path(original_path):
    original_path = str(original_path)

    if Path(original_path).exists():
        return original_path

    normalized = original_path.replace("\\", "/")
    marker = "/1.데이터/"

    if marker in normalized:
        relative_path = normalized.split(
            marker,
            maxsplit=1,
        )[1]

        candidate = DRIVE_DATA_ROOT / relative_path

        if candidate.exists():
            return str(candidate)

    return original_path


def read_jsonl_with_resolved_paths(path):
    records = []

    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as file:
        for line in file:
            if not line.strip():
                continue

            record = json.loads(line)
            record["image_path"] = resolve_image_path(
                record["image_path"]
            )
            records.append(record)

    return records


train_records = read_jsonl_with_resolved_paths(
    FULL_TRAIN_MANIFEST
)

val_records = read_jsonl_with_resolved_paths(
    FULL_VAL_MANIFEST
)

test_records = read_jsonl_with_resolved_paths(
    FULL_TEST_MANIFEST
)

print(len(train_records), len(val_records), len(test_records))
print(Path(test_records[0]["image_path"]).exists())

53672 10052 8501
True


In [11]:
full_test_dataset = PPEPersonROIDataset(
    records=test_records,
    image_size=512,
    augment=False,
)

full_test_loader = DataLoader(
    full_test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

In [12]:
full_model = SharedBackbonePPEModel(
    fpn_channels=128,
    pretrained=False,
).to(device)

checkpoint = torch.load(
    FINAL_CHECKPOINT_PATH,
    map_location=device,
)

full_model.load_state_dict(
    checkpoint["model_state_dict"]
)

full_model.eval()

print("checkpoint epoch:", checkpoint["epoch"])
print("최종 모델 로드 완료")

checkpoint epoch: 4
최종 모델 로드 완료


In [13]:
RESULT_ROOT = (
    FULL_CHECKPOINT_ROOT
    / "evaluation"
)

test_prediction_df = pd.read_csv(
    RESULT_ROOT
    / "test_status_predictions.csv"
)

test_class_results = pd.read_csv(
    RESULT_ROOT
    / "test_class_metrics.csv"
)

misclassified_df = pd.read_csv(
    RESULT_ROOT
    / "test_misclassified.csv"
)

print("예측:", len(test_prediction_df))
print("오분류:", len(misclassified_df))
display(test_class_results)

예측: 8501
오분류: 254


,class_name,count,dice,iou,accuracy,tp,fp,tn,fn
0,harness_off,1857,0.900234,0.825957,1.000000,0,0,1857,0
1,harness_on,1815,0.763750,0.634600,0.997796,1811,0,0,4
2,helmet_off,1434,0.948613,0.903656,0.999303,0,1,1433,0
3,helmet_on,1449,0.802712,0.736451,0.828157,1200,0,0,249
4,welding_mask_off,1420,0.935845,0.881293,1.000000,0,0,1420,0
5,welding_mask_on,526,0.909016,0.837630,1.000000,526,0,0,0


In [14]:
helmet_on_errors = test_prediction_df[
    (test_prediction_df["task_name"] == "helmet")
    & (test_prediction_df["true_label"] == 1)
    & (test_prediction_df["pred_label"] == 0)
].copy()

test_record_map = {
    str(record["record_id"]): record
    for record in test_records
}

print("helmet_on → off:", len(helmet_on_errors))

helmet_on → off: 249


In [17]:
show_helmet_on_errors(
    error_df=helmet_on_errors,
    record_map=test_record_map,
    model=full_model,
    dataset=full_test_dataset,
    device=device,
    start_index=0,
    count=12,
)

Output hidden; open in https://colab.research.google.com to view.

In [18]:
def get_session_id(record_id):
    parts = str(record_id).split("_")
    return "_".join(parts[:2])


rows = []

for split_name, records in [
    ("train", train_records),
    ("val", val_records),
    ("test", test_records),
]:
    for record in records:
        session_id = get_session_id(
            record["record_id"]
        )

        if session_id != "S63_DATA3":
            continue

        if record["class_name"] not in [
            "helmet_off",
            "helmet_on",
        ]:
            continue

        rows.append({
            "split": split_name,
            "class_name": record["class_name"],
            "record_id": record["record_id"],
        })


s63_distribution = (
    pd.DataFrame(rows)
    .groupby(["split", "class_name"])
    .size()
    .reset_index(name="count")
)

display(s63_distribution)

,split,class_name,count
0,test,helmet_off,1434
1,test,helmet_on,1449
2,train,helmet_off,11975
3,train,helmet_on,10367
4,val,helmet_off,1362
5,val,helmet_on,2424


In [19]:
s63_predictions = test_prediction_df[
    test_prediction_df["record_id"]
    .astype(str)
    .str.startswith("S63_DATA3")
].copy()

display(
    s63_predictions.groupby(
        ["class_name", "true_label", "pred_label"]
    )
    .size()
    .reset_index(name="count")
)

,class_name,true_label,pred_label,count
0,harness_off,0,0,1857
1,harness_on,1,0,4
2,harness_on,1,1,1811
3,helmet_off,0,0,1433
4,helmet_off,0,1,1
5,helmet_on,1,0,249
6,helmet_on,1,1,1200
7,welding_mask_off,0,0,1420
8,welding_mask_on,1,1,526


In [20]:
helmet_on_errors["frame_number"] = (
    helmet_on_errors["record_id"]
    .astype(str)
    .str.extract(r"_(\d{6})__")
    .astype(float)
)

display(
    helmet_on_errors[
        ["record_id", "frame_number", "prob_off"]
    ].head(30)
)

,record_id,frame_number,prob_off
5215,S63_DATA3_H1_L1_D2023-09-14-10-23_001_000372__...,372.0,0.810741
5216,S63_DATA3_H1_L1_D2023-09-14-10-23_001_000374__...,374.0,0.914291
5217,S63_DATA3_H1_L1_D2023-09-14-10-23_001_000375__...,375.0,0.908861
5218,S63_DATA3_H1_L1_D2023-09-14-10-23_001_000380__...,380.0,0.937555
5219,S63_DATA3_H1_L1_D2023-09-14-10-23_001_000382__...,382.0,0.869604
5220,S63_DATA3_H1_L1_D2023-09-14-10-23_001_000383__...,383.0,0.610163
5266,S63_DATA3_H1_L1_D2023-09-14-10-23_001_000723__...,723.0,0.638152
5274,S63_DATA3_H1_L1_D2023-09-14-10-23_001_000735__...,735.0,0.853242
5275,S63_DATA3_H1_L1_D2023-09-14-10-23_001_000739__...,739.0,0.998283
5276,S63_DATA3_H1_L1_D2023-09-14-10-23_001_000740__...,740.0,0.925634


In [21]:
frame_numbers = (
    helmet_on_errors["frame_number"]
    .dropna()
    .sort_values()
    .astype(int)
    .tolist()
)

frame_gaps = [
    current - previous
    for previous, current
    in zip(frame_numbers[:-1], frame_numbers[1:])
]

print("오류 프레임 수:", len(frame_numbers))
print("프레임 간격 1 이하:", sum(gap <= 1 for gap in frame_gaps))
print("프레임 간격 5 이하:", sum(gap <= 5 for gap in frame_gaps))

오류 프레임 수: 249
프레임 간격 1 이하: 145
프레임 간격 5 이하: 203


In [22]:
frame_numbers = (
    helmet_on_errors["frame_number"]
    .dropna()
    .sort_values()
    .astype(int)
    .tolist()
)

frame_gaps = [
    current - previous
    for previous, current
    in zip(frame_numbers[:-1], frame_numbers[1:])
]

print("오류 프레임 수:", len(frame_numbers))
print("프레임 간격 1 이하:", sum(gap <= 1 for gap in frame_gaps))
print("프레임 간격 5 이하:", sum(gap <= 5 for gap in frame_gaps))

오류 프레임 수: 249
프레임 간격 1 이하: 145
프레임 간격 5 이하: 203


In [23]:
helmet_on_error_ids = set(
    helmet_on_errors["record_id"].astype(str)
)

helmet_on_error_metrics = test_class_results[
    test_class_results["class_name"]
    == "helmet_on"
]

display(helmet_on_error_metrics)

,class_name,count,dice,iou,accuracy,tp,fp,tn,fn
3,helmet_on,1449,0.802712,0.736451,0.828157,1200,0,0,249


In [24]:
helmet_on_errors["sequence_id"] = (
    helmet_on_errors["record_id"]
    .astype(str)
    .str.extract(
        r"^(.*)_\d{6}__"
    )[0]
)

sequence_error_summary = (
    helmet_on_errors
    .groupby("sequence_id")
    .agg(
        error_count=("record_id", "count"),
        mean_prob_off=("prob_off", "mean"),
        min_frame=("frame_number", "min"),
        max_frame=("frame_number", "max"),
    )
    .sort_values(
        "error_count",
        ascending=False,
    )
    .reset_index()
)

print(
    "오류가 발생한 실제 시퀀스 수:",
    helmet_on_errors["sequence_id"].nunique(),
)

display(
    sequence_error_summary.head(20)
)

오류가 발생한 실제 시퀀스 수: 3


,sequence_id,error_count,mean_prob_off,min_frame,max_frame
0,S63_DATA3_H1_L1_D2023-10-12-10-16_001,142,0.699302,1043.0,1813.0
1,S63_DATA3_H1_L1_D2023-09-14-10-23_001,82,0.833109,372.0,2487.0
2,S63_DATA3_H1_L1_D2023-10-13-10-20_001,25,0.633801,14.0,516.0


In [25]:
error_runs = []

for sequence_id, group in helmet_on_errors.groupby(
    "sequence_id"
):
    group = group.sort_values(
        "frame_number"
    ).copy()

    group["frame_gap"] = (
        group["frame_number"].diff()
    )

    group["run_id"] = (
        group["frame_gap"]
        .fillna(999)
        .gt(5)
        .cumsum()
    )

    for run_id, run_group in group.groupby(
        "run_id"
    ):
        error_runs.append({
            "sequence_id": sequence_id,
            "run_id": int(run_id),
            "start_frame": int(
                run_group["frame_number"].min()
            ),
            "end_frame": int(
                run_group["frame_number"].max()
            ),
            "frame_count": len(run_group),
            "mean_prob_off": (
                run_group["prob_off"].mean()
            ),
            "max_prob_off": (
                run_group["prob_off"].max()
            ),
        })

error_run_df = (
    pd.DataFrame(error_runs)
    .sort_values(
        "frame_count",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(
    "249개 오류가 속한 독립 오류 구간:",
    len(error_run_df),
)

display(
    error_run_df.head(30).round(4)
)

249개 오류가 속한 독립 오류 구간: 48


,sequence_id,run_id,start_frame,end_frame,frame_count,mean_prob_off,max_prob_off
0,S63_DATA3_H1_L1_D2023-10-12-10-16_001,7,1383,1424,25,0.6164,0.7759
1,S63_DATA3_H1_L1_D2023-10-12-10-16_001,19,1750,1774,22,0.9117,0.9974
2,S63_DATA3_H1_L1_D2023-09-14-10-23_001,5,771,794,18,0.8768,0.9731
3,S63_DATA3_H1_L1_D2023-09-14-10-23_001,3,735,754,15,0.8723,0.9983
4,S63_DATA3_H1_L1_D2023-10-12-10-16_001,8,1430,1451,15,0.5901,0.7712
5,S63_DATA3_H1_L1_D2023-10-12-10-16_001,9,1461,1481,14,0.5923,0.6784
6,S63_DATA3_H1_L1_D2023-10-12-10-16_001,16,1695,1710,14,0.6823,0.9191
7,S63_DATA3_H1_L1_D2023-10-12-10-16_001,12,1566,1581,11,0.8686,0.9928
8,S63_DATA3_H1_L1_D2023-10-13-10-20_001,2,24,32,9,0.7030,0.7961
9,S63_DATA3_H1_L1_D2023-10-12-10-16_001,20,1796,1804,8,0.8045,0.9933


In [26]:
representative_rows = []

for _, run in error_run_df.iterrows():
    sequence_id = run["sequence_id"]
    start_frame = run["start_frame"]
    end_frame = run["end_frame"]

    candidates = helmet_on_errors[
        (helmet_on_errors["sequence_id"] == sequence_id)
        & (helmet_on_errors["frame_number"] >= start_frame)
        & (helmet_on_errors["frame_number"] <= end_frame)
    ].copy()

    if len(candidates) == 0:
        continue

    representative = candidates.sort_values(
        "prob_off",
        ascending=False,
    ).iloc[0]

    representative_rows.append(
        representative
    )

representative_error_df = pd.DataFrame(
    representative_rows
).reset_index(drop=True)

print("대표 오류 장면 수:", len(representative_error_df))

display(
    representative_error_df[
        [
            "record_id",
            "sequence_id",
            "frame_number",
            "prob_off",
        ]
    ].head(20)
)

대표 오류 장면 수: 48


,record_id,sequence_id,frame_number,prob_off
0,S63_DATA3_H1_L1_D2023-10-12-10-16_001_001396__...,S63_DATA3_H1_L1_D2023-10-12-10-16_001,1396.0,0.775942
1,S63_DATA3_H1_L1_D2023-10-12-10-16_001_001772__...,S63_DATA3_H1_L1_D2023-10-12-10-16_001,1772.0,0.997373
2,S63_DATA3_H1_L1_D2023-09-14-10-23_001_000781__...,S63_DATA3_H1_L1_D2023-09-14-10-23_001,781.0,0.973062
3,S63_DATA3_H1_L1_D2023-09-14-10-23_001_000739__...,S63_DATA3_H1_L1_D2023-09-14-10-23_001,739.0,0.998283
4,S63_DATA3_H1_L1_D2023-10-12-10-16_001_001438__...,S63_DATA3_H1_L1_D2023-10-12-10-16_001,1438.0,0.771201
5,S63_DATA3_H1_L1_D2023-10-12-10-16_001_001481__...,S63_DATA3_H1_L1_D2023-10-12-10-16_001,1481.0,0.678407
6,S63_DATA3_H1_L1_D2023-10-12-10-16_001_001697__...,S63_DATA3_H1_L1_D2023-10-12-10-16_001,1697.0,0.919100
7,S63_DATA3_H1_L1_D2023-10-12-10-16_001_001581__...,S63_DATA3_H1_L1_D2023-10-12-10-16_001,1581.0,0.992837
8,S63_DATA3_H1_L1_D2023-10-13-10-20_001_000028__...,S63_DATA3_H1_L1_D2023-10-13-10-20_001,28.0,0.796124
9,S63_DATA3_H1_L1_D2023-10-12-10-16_001_001797__...,S63_DATA3_H1_L1_D2023-10-12-10-16_001,1797.0,0.993301


In [27]:
show_helmet_on_errors(
    error_df=representative_error_df,
    record_map=test_record_map,
    model=full_model,
    dataset=full_test_dataset,
    device=device,
    start_index=0,
    count=12,
)

Output hidden; open in https://colab.research.google.com to view.

In [28]:
class SharedBackbonePPEModel(nn.Module):
    def __init__(
        self,
        fpn_channels=128,
        pretrained=True,
        num_tasks=3,
        helmet_task_index=1,
        helmet_mask_weight=0.8,
    ):
        super().__init__()

        self.helmet_task_index = helmet_task_index
        self.helmet_mask_weight = helmet_mask_weight

        weights = (
            ResNet18_Weights.DEFAULT
            if pretrained
            else None
        )

        backbone = resnet18(
            weights=weights
        )

        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
        )

        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.lateral2 = nn.Conv2d(
            64,
            fpn_channels,
            kernel_size=1,
        )

        self.lateral3 = nn.Conv2d(
            128,
            fpn_channels,
            kernel_size=1,
        )

        self.lateral4 = nn.Conv2d(
            256,
            fpn_channels,
            kernel_size=1,
        )

        self.lateral5 = nn.Conv2d(
            512,
            fpn_channels,
            kernel_size=1,
        )

        self.smooth2 = nn.Conv2d(
            fpn_channels,
            fpn_channels,
            kernel_size=3,
            padding=1,
        )

        self.segmentation_heads = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(
                    fpn_channels,
                    fpn_channels,
                    kernel_size=3,
                    padding=1,
                ),
                nn.BatchNorm2d(
                    fpn_channels
                ),
                nn.ReLU(inplace=True),

                nn.Conv2d(
                    fpn_channels,
                    64,
                    kernel_size=3,
                    padding=1,
                ),
                nn.ReLU(inplace=True),

                nn.Conv2d(
                    64,
                    1,
                    kernel_size=1,
                ),
            )
            for _ in range(num_tasks)
        ])

        self.global_pool = nn.AdaptiveAvgPool2d(
            (1, 1)
        )

        self.status_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(512, 128),
                nn.ReLU(inplace=True),
                nn.Dropout(0.2),
                nn.Linear(128, 2),
            )
            for _ in range(num_tasks)
        ])

    def masked_feature_pool(
        self,
        feature_map,
        segmentation_logits,
        eps=1e-6,
    ):
        """
        segmentation mask가 가리키는 영역을 중심으로
        c5 feature를 가중 평균한다.

        feature_map:
            [B, 512, Hc, Wc]

        segmentation_logits:
            [B, 1, Hs, Ws]
        """

        attention_mask = torch.sigmoid(
            segmentation_logits
        )

        attention_mask = F.interpolate(
            attention_mask,
            size=feature_map.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        # 분류 학습이 segmentation 결과를 역으로
        # 흔들지 않도록 gradient 차단
        attention_mask = attention_mask.detach()

        weighted_features = (
            feature_map * attention_mask
        ).sum(dim=(2, 3))

        mask_sum = attention_mask.sum(
            dim=(2, 3)
        )

        masked_pooled = (
            weighted_features
            / mask_sum.clamp_min(eps)
        )

        return masked_pooled

    def forward(
        self,
        images,
        task_indices,
    ):
        x = self.stem(images)

        c2 = self.layer1(x)
        c3 = self.layer2(c2)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)

        p5 = self.lateral5(c5)

        p4 = (
            self.lateral4(c4)
            + F.interpolate(
                p5,
                size=c4.shape[-2:],
                mode="nearest",
            )
        )

        p3 = (
            self.lateral3(c3)
            + F.interpolate(
                p4,
                size=c3.shape[-2:],
                mode="nearest",
            )
        )

        p2 = (
            self.lateral2(c2)
            + F.interpolate(
                p3,
                size=c2.shape[-2:],
                mode="nearest",
            )
        )

        p2 = self.smooth2(p2)

        batch_size = images.shape[0]

        seg_logits = torch.empty(
            (
                batch_size,
                1,
                images.shape[-2],
                images.shape[-1],
            ),
            device=images.device,
            dtype=torch.float32,
        )

        status_logits = torch.empty(
            (batch_size, 2),
            device=images.device,
            dtype=torch.float32,
        )

        global_features = (
            self.global_pool(c5)
            .flatten(1)
        )

        for task_index in range(
            len(self.segmentation_heads)
        ):
            sample_mask = (
                task_indices == task_index
            )

            if not sample_mask.any():
                continue

            selected_p2 = p2[sample_mask]
            selected_c5 = c5[sample_mask]

            # 각 PPE 전용 segmentation
            task_seg_logits_low_resolution = (
                self.segmentation_heads[
                    task_index
                ](
                    selected_p2
                )
            )

            # helmet task는 segmentation 위치 중심 feature 사용
            if task_index == self.helmet_task_index:
                masked_features = (
                    self.masked_feature_pool(
                        feature_map=selected_c5,
                        segmentation_logits=(
                            task_seg_logits_low_resolution
                        ),
                    )
                )

                selected_global_features = (
                    global_features[sample_mask]
                )

                # 마스크 영역 80% + 전체 문맥 20%
                classification_features = (
                    self.helmet_mask_weight
                    * masked_features
                    + (
                        1.0
                        - self.helmet_mask_weight
                    )
                    * selected_global_features
                )

            else:
                classification_features = (
                    global_features[sample_mask]
                )

            task_status_logits = (
                self.status_heads[
                    task_index
                ](
                    classification_features
                )
            )

            task_seg_logits = F.interpolate(
                task_seg_logits_low_resolution,
                size=images.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )

            seg_logits[sample_mask] = (
                task_seg_logits.float()
            )

            status_logits[sample_mask] = (
                task_status_logits.float()
            )

        return {
            "seg_logits": seg_logits,
            "status_logits": status_logits,
        }

In [29]:
print(TASK_TO_INDEX)

{'helmet': 0, 'harness': 1, 'welding_mask': 2}


In [30]:
HELMET_TASK_INDEX = TASK_TO_INDEX["helmet"]

print(
    "helmet task index:",
    HELMET_TASK_INDEX,
)

helmet task index: 0


In [31]:
improved_model = SharedBackbonePPEModel(
    fpn_channels=128,
    pretrained=False,
    num_tasks=3,
    helmet_task_index=HELMET_TASK_INDEX,
    helmet_mask_weight=0.8,
).to(device)

In [32]:
baseline_checkpoint = torch.load(
    FINAL_CHECKPOINT_PATH,
    map_location=device,
)

load_result = improved_model.load_state_dict(
    baseline_checkpoint[
        "model_state_dict"
    ],
    strict=True,
)

print(load_result)
print(
    "불러온 epoch:",
    baseline_checkpoint["epoch"],
)

<All keys matched successfully>
불러온 epoch: 4


In [35]:
improved_model.eval()

pretrain_improved_results = (
    evaluate_by_class(
        model=improved_model,
        loader=full_test_loader,
        device=device,
    )
)

display(
    pretrain_improved_results.round(4)
)

Class-wise test:   0%|          | 0/266 [00:00<?, ?it/s]

,class_name,count,dice,iou,accuracy,tp,fp,tn,fn
0,harness_off,1857,0.9019,0.8286,1.0000,0,0,1857,0
1,harness_on,1815,0.7678,0.6405,0.9978,1811,0,0,4
2,helmet_off,1434,0.9502,0.9064,1.0000,0,0,1434,0
3,helmet_on,1449,0.8053,0.7408,0.7543,1093,0,0,356
4,welding_mask_off,1420,0.9379,0.8849,1.0000,0,0,1420,0
5,welding_mask_on,526,0.9137,0.8454,1.0000,526,0,0,0


In [40]:
pretrain_val_results = evaluate_by_class(
    model=improved_model,
    loader=full_val_loader,
    device=device,
)

display(
    pretrain_val_results.round(4)
)

Class-wise test:   0%|          | 0/315 [00:00<?, ?it/s]

,class_name,count,dice,iou,accuracy,tp,fp,tn,fn
0,harness_off,2116,0.9101,0.8401,1.0000,0,0,2116,0
1,harness_on,1822,0.7772,0.6562,0.9984,1819,0,0,3
2,helmet_off,1362,0.9198,0.8649,0.8744,0,171,1191,0
3,helmet_on,2424,0.8659,0.7909,0.9917,2404,0,0,20
4,welding_mask_off,1150,0.9545,0.9147,1.0000,0,0,1150,0
5,welding_mask_on,1178,0.9312,0.8753,1.0000,1178,0,0,0


In [41]:
for parameter in improved_model.parameters():
    parameter.requires_grad = False

In [42]:
for parameter in improved_model.status_heads[
    HELMET_TASK_INDEX
].parameters():
    parameter.requires_grad = True

In [43]:
trainable_parameters = [
    name
    for name, parameter
    in improved_model.named_parameters()
    if parameter.requires_grad
]

print(
    "학습 대상 파라미터:"
)

for name in trainable_parameters:
    print(name)

학습 대상 파라미터:
status_heads.0.0.weight
status_heads.0.0.bias
status_heads.0.3.weight
status_heads.0.3.bias


In [44]:
helmet_optimizer = torch.optim.AdamW(
    filter(
        lambda parameter:
            parameter.requires_grad,
        improved_model.parameters(),
    ),
    lr=1e-4,
    weight_decay=1e-4,
)

In [46]:
HELMET_TASK_INDEX = TASK_TO_INDEX["helmet"]

helmet_train_records = [
    record
    for record in train_records
    if record["class_name"] in {
        "helmet_off",
        "helmet_on",
    }
]

helmet_val_records = [
    record
    for record in val_records
    if record["class_name"] in {
        "helmet_off",
        "helmet_on",
    }
]

print("Helmet train:", len(helmet_train_records))
print("Helmet val:", len(helmet_val_records))

print(
    pd.Series(
        record["class_name"]
        for record in helmet_train_records
    ).value_counts()
)

print(
    pd.Series(
        record["class_name"]
        for record in helmet_val_records
    ).value_counts()
)

Helmet train: 22342
Helmet val: 3786
helmet_off    11975
helmet_on     10367
Name: count, dtype: int64
helmet_on     2424
helmet_off    1362
Name: count, dtype: int64


In [47]:
helmet_train_dataset = PPEPersonROIDataset(
    records=helmet_train_records,
    image_size=512,
    augment=True,
)

helmet_val_dataset = PPEPersonROIDataset(
    records=helmet_val_records,
    image_size=512,
    augment=False,
)

In [48]:
helmet_train_loader = DataLoader(
    helmet_train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

helmet_val_loader = DataLoader(
    helmet_val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

print("Train batches:", len(helmet_train_loader))
print("Val batches:", len(helmet_val_loader))

Train batches: 699
Val batches: 119


In [49]:
sample_batch = next(
    iter(helmet_train_loader)
)

print(
    "task indices:",
    torch.unique(
        sample_batch["task_index"]
    )
)

print(
    "status labels:",
    torch.unique(
        sample_batch["status_label"]
    )
)

print(
    "class names:",
    set(sample_batch["class_name"])
)

task indices: tensor([0])
status labels: tensor([0, 1])
class names: {'helmet_on', 'helmet_off'}


In [50]:
for parameter in improved_model.parameters():
    parameter.requires_grad = False

for parameter in improved_model.status_heads[
    HELMET_TASK_INDEX
].parameters():
    parameter.requires_grad = True

In [51]:
for name, parameter in improved_model.named_parameters():
    if parameter.requires_grad:
        print(name)

status_heads.0.0.weight
status_heads.0.0.bias
status_heads.0.3.weight
status_heads.0.3.bias


In [52]:
helmet_criterion = nn.CrossEntropyLoss()

helmet_optimizer = torch.optim.AdamW(
    improved_model.status_heads[
        HELMET_TASK_INDEX
    ].parameters(),
    lr=1e-4,
    weight_decay=1e-4,
)

In [53]:
def train_helmet_status_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
):
    # backbone과 segmentation의 BN/Dropout은 고정
    model.eval()

    # helmet status head의 Dropout만 학습 모드
    model.status_heads[
        HELMET_TASK_INDEX
    ].train()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    true_off = 0
    correct_off = 0

    true_on = 0
    correct_on = 0

    progress_bar = tqdm(
        loader,
        desc="Train helmet head",
    )

    for batch in progress_bar:
        images = batch["image"].to(
            device,
            non_blocking=True,
        )

        task_indices = batch[
            "task_index"
        ].to(
            device,
            non_blocking=True,
        )

        status_labels = batch[
            "status_label"
        ].to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=True,
        ):
            outputs = model(
                images,
                task_indices,
            )

            loss = criterion(
                outputs["status_logits"],
                status_labels,
            )

        loss.backward()
        optimizer.step()

        predictions = (
            outputs["status_logits"]
            .argmax(dim=1)
        )

        batch_size = images.size(0)

        total_loss += (
            loss.item() * batch_size
        )

        total_correct += (
            predictions
            .eq(status_labels)
            .sum()
            .item()
        )

        total_count += batch_size

        off_mask = status_labels == 0
        on_mask = status_labels == 1

        true_off += off_mask.sum().item()
        true_on += on_mask.sum().item()

        correct_off += (
            predictions[off_mask]
            .eq(0)
            .sum()
            .item()
        )

        correct_on += (
            predictions[on_mask]
            .eq(1)
            .sum()
            .item()
        )

        progress_bar.set_postfix({
            "loss": (
                total_loss
                / max(total_count, 1)
            ),
            "acc": (
                total_correct
                / max(total_count, 1)
            ),
        })

    return {
        "loss": (
            total_loss
            / max(total_count, 1)
        ),
        "accuracy": (
            total_correct
            / max(total_count, 1)
        ),
        "off_recall": (
            correct_off
            / max(true_off, 1)
        ),
        "on_recall": (
            correct_on
            / max(true_on, 1)
        ),
    }

In [54]:
@torch.no_grad()
def validate_helmet_status(
    model,
    loader,
    criterion,
    device,
):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    true_off = 0
    correct_off = 0

    true_on = 0
    correct_on = 0

    all_true = []
    all_pred = []

    for batch in tqdm(
        loader,
        desc="Validate helmet head",
    ):
        images = batch["image"].to(
            device,
            non_blocking=True,
        )

        task_indices = batch[
            "task_index"
        ].to(
            device,
            non_blocking=True,
        )

        status_labels = batch[
            "status_label"
        ].to(
            device,
            non_blocking=True,
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=True,
        ):
            outputs = model(
                images,
                task_indices,
            )

            loss = criterion(
                outputs["status_logits"],
                status_labels,
            )

        predictions = (
            outputs["status_logits"]
            .argmax(dim=1)
        )

        batch_size = images.size(0)

        total_loss += (
            loss.item() * batch_size
        )

        total_correct += (
            predictions
            .eq(status_labels)
            .sum()
            .item()
        )

        total_count += batch_size

        off_mask = status_labels == 0
        on_mask = status_labels == 1

        true_off += off_mask.sum().item()
        true_on += on_mask.sum().item()

        correct_off += (
            predictions[off_mask]
            .eq(0)
            .sum()
            .item()
        )

        correct_on += (
            predictions[on_mask]
            .eq(1)
            .sum()
            .item()
        )

        all_true.extend(
            status_labels.cpu().tolist()
        )

        all_pred.extend(
            predictions.cpu().tolist()
        )

    off_recall = (
        correct_off
        / max(true_off, 1)
    )

    on_recall = (
        correct_on
        / max(true_on, 1)
    )

    balanced_accuracy = (
        off_recall + on_recall
    ) / 2.0

    return {
        "loss": (
            total_loss
            / max(total_count, 1)
        ),
        "accuracy": (
            total_correct
            / max(total_count, 1)
        ),
        "off_recall": off_recall,
        "on_recall": on_recall,
        "balanced_accuracy":
            balanced_accuracy,
        "true_labels": all_true,
        "pred_labels": all_pred,
    }

In [55]:
HELMET_IMPROVEMENT_ROOT = (
    FULL_CHECKPOINT_ROOT
    / "helmet_mask_guided"
)

HELMET_IMPROVEMENT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

BEST_HELMET_PATH = (
    HELMET_IMPROVEMENT_ROOT
    / "best_helmet_status_model.pt"
)

print(HELMET_IMPROVEMENT_ROOT)

/content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/checkpoints/full_hybrid_person_roi/helmet_mask_guided


In [56]:
NUM_HELMET_EPOCHS = 5

best_balanced_accuracy = -1.0
helmet_training_history = []

for epoch in range(
    1,
    NUM_HELMET_EPOCHS + 1,
):
    train_metrics = (
        train_helmet_status_one_epoch(
            model=improved_model,
            loader=helmet_train_loader,
            optimizer=helmet_optimizer,
            criterion=helmet_criterion,
            device=device,
        )
    )

    val_metrics = validate_helmet_status(
        model=improved_model,
        loader=helmet_val_loader,
        criterion=helmet_criterion,
        device=device,
    )

    row = {
        "epoch": epoch,
        "train_loss":
            train_metrics["loss"],
        "train_accuracy":
            train_metrics["accuracy"],
        "train_off_recall":
            train_metrics["off_recall"],
        "train_on_recall":
            train_metrics["on_recall"],
        "val_loss":
            val_metrics["loss"],
        "val_accuracy":
            val_metrics["accuracy"],
        "val_off_recall":
            val_metrics["off_recall"],
        "val_on_recall":
            val_metrics["on_recall"],
        "val_balanced_accuracy":
            val_metrics[
                "balanced_accuracy"
            ],
    }

    helmet_training_history.append(row)

    print(
        f"\nEpoch {epoch}/{NUM_HELMET_EPOCHS}"
    )

    print(
        f"Train loss={train_metrics['loss']:.4f} | "
        f"acc={train_metrics['accuracy']:.4f} | "
        f"OFF recall={train_metrics['off_recall']:.4f} | "
        f"ON recall={train_metrics['on_recall']:.4f}"
    )

    print(
        f"Val loss={val_metrics['loss']:.4f} | "
        f"acc={val_metrics['accuracy']:.4f} | "
        f"OFF recall={val_metrics['off_recall']:.4f} | "
        f"ON recall={val_metrics['on_recall']:.4f} | "
        f"balanced acc="
        f"{val_metrics['balanced_accuracy']:.4f}"
    )

    if (
        val_metrics["balanced_accuracy"]
        > best_balanced_accuracy
    ):
        best_balanced_accuracy = (
            val_metrics[
                "balanced_accuracy"
            ]
        )

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict":
                    improved_model.state_dict(),
                "optimizer_state_dict":
                    helmet_optimizer.state_dict(),
                "val_metrics": {
                    key: value
                    for key, value
                    in val_metrics.items()
                    if key not in {
                        "true_labels",
                        "pred_labels",
                    }
                },
                "helmet_task_index":
                    HELMET_TASK_INDEX,
                "helmet_mask_weight":
                    improved_model
                    .helmet_mask_weight,
            },
            BEST_HELMET_PATH,
        )

        print(
            "Best helmet model 저장:",
            BEST_HELMET_PATH,
        )

Train helmet head:   0%|          | 0/699 [00:00<?, ?it/s]

Validate helmet head:   0%|          | 0/119 [00:00<?, ?it/s]


Epoch 1/5
Train loss=0.0002 | acc=1.0000 | OFF recall=0.9999 | ON recall=1.0000
Val loss=0.1954 | acc=0.9614 | OFF recall=0.9295 | ON recall=0.9794 | balanced acc=0.9544
Best helmet model 저장: /content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/checkpoints/full_hybrid_person_roi/helmet_mask_guided/best_helmet_status_model.pt


Train helmet head:   0%|          | 0/699 [00:00<?, ?it/s]

Validate helmet head:   0%|          | 0/119 [00:00<?, ?it/s]

...

Validate helmet head:   0%|          | 0/119 [00:00<?, ?it/s]


Epoch 4/5
Train loss=0.0000 | acc=1.0000 | OFF recall=1.0000 | ON recall=1.0000
Val loss=0.3031 | acc=0.9548 | OFF recall=0.8891 | ON recall=0.9917 | balanced acc=0.9404


Train helmet head:   0%|          | 0/699 [00:00<?, ?it/s]

Validate helmet head:   0%|          | 0/119 [00:00<?, ?it/s]


Epoch 5/5
Train loss=0.0009 | acc=0.9999 | OFF recall=0.9998 | ON recall=0.9999
Val loss=0.1277 | acc=0.9712 | OFF recall=0.9413 | ON recall=0.9880 | balanced acc=0.9646
Best helmet model 저장: /content/drive/MyDrive/YardOps_PPE_Dataset/shared_backbone_workspace/checkpoints/full_hybrid_person_roi/helmet_mask_guided/best_helmet_status_model.pt


In [57]:
helmet_history_df = pd.DataFrame(
    helmet_training_history
)

helmet_history_df.to_csv(
    HELMET_IMPROVEMENT_ROOT
    / "helmet_training_history.csv",
    index=False,
    encoding="utf-8-sig",
)

display(
    helmet_history_df.round(4)
)

,epoch,train_loss,train_accuracy,train_off_recall,train_on_recall,val_loss,val_accuracy,val_off_recall,val_on_recall,val_balanced_accuracy
0,1,0.0002,1.0000,0.9999,1.0000,0.1954,0.9614,0.9295,0.9794,0.9544
1,2,0.0001,1.0000,1.0000,0.9999,0.3266,0.9525,0.8811,0.9926,0.9368
2,3,0.0000,1.0000,1.0000,1.0000,0.2838,0.9546,0.8913,0.9901,0.9407
3,4,0.0000,1.0000,1.0000,1.0000,0.3031,0.9548,0.8891,0.9917,0.9404
4,5,0.0009,0.9999,0.9998,0.9999,0.1277,0.9712,0.9413,0.9880,0.9646


In [58]:
helmet_checkpoint = torch.load(
    BEST_HELMET_PATH,
    map_location=device,
)

improved_model.load_state_dict(
    helmet_checkpoint["model_state_dict"]
)

improved_model.eval()

improved_test_results = evaluate_by_class(
    model=improved_model,
    loader=full_test_loader,
    device=device,
)

display(
    improved_test_results.round(4)
)

Class-wise test:   0%|          | 0/266 [00:00<?, ?it/s]

,class_name,count,dice,iou,accuracy,tp,fp,tn,fn
0,harness_off,1857,0.9019,0.8286,1.0000,0,0,1857,0
1,harness_on,1815,0.7678,0.6405,0.9978,1811,0,0,4
2,helmet_off,1434,0.9502,0.9064,1.0000,0,0,1434,0
3,helmet_on,1449,0.8053,0.7408,0.6529,946,0,0,503
4,welding_mask_off,1420,0.9379,0.8849,1.0000,0,0,1420,0
5,welding_mask_on,526,0.9137,0.8454,1.0000,526,0,0,0


In [59]:
baseline_helmet = test_class_results[
    test_class_results["class_name"].isin([
        "helmet_off",
        "helmet_on",
    ])
].copy()

improved_helmet = improved_test_results[
    improved_test_results["class_name"].isin([
        "helmet_off",
        "helmet_on",
    ])
].copy()

comparison_df = baseline_helmet[
    ["class_name", "dice", "iou", "accuracy"]
].merge(
    improved_helmet[
        ["class_name", "dice", "iou", "accuracy"]
    ],
    on="class_name",
    suffixes=("_baseline", "_improved"),
)

display(comparison_df.round(4))

,class_name,dice_baseline,iou_baseline,accuracy_baseline,dice_improved,iou_improved,accuracy_improved
0,helmet_off,0.9486,0.9037,0.9993,0.9502,0.9064,1.0000
1,helmet_on,0.8027,0.7365,0.8282,0.8053,0.7408,0.6529


In [60]:
baseline_checkpoint = torch.load(
    FINAL_CHECKPOINT_PATH,
    map_location=device,
)

full_model = SharedBackbonePPEModel(
    fpn_channels=128,
    pretrained=False,
    num_tasks=3,
    helmet_task_index=0,
    helmet_mask_weight=0.8,
).to(device)

full_model.load_state_dict(
    baseline_checkpoint["model_state_dict"],
    strict=True,
)

full_model.eval()

print("Baseline 복구 완료")
print("checkpoint epoch:", baseline_checkpoint["epoch"])

Baseline 복구 완료
checkpoint epoch: 4


In [62]:
improved_model.helmet_mask_weight = 0.0

In [63]:
full_model.helmet_mask_weight = 0.0
full_model.eval()

SharedBackbonePPEModel(
  (stem): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


In [64]:
restored_results = evaluate_by_class(
    model=full_model,
    loader=full_test_loader,
    device=device,
)

display(
    restored_results[
        restored_results["class_name"].isin(
            ["helmet_off", "helmet_on"]
        )
    ].round(4)
)

Class-wise test:   0%|          | 0/266 [00:00<?, ?it/s]

,class_name,count,dice,iou,accuracy,tp,fp,tn,fn
2,helmet_off,1434,0.9502,0.9064,0.9993,0,1,1433,0
3,helmet_on,1449,0.8053,0.7408,0.8282,1200,0,0,249
